# 03 - Build the ENS035 / PQENS087 Common Master

            Run all cells to rebuild the common earnings-risk master from the frozen real quarterly Vietstock data. The notebook embeds the accepted 2026-06-19 release code and writes all regenerated artifacts under `GENERATED_OUTPUTS/03_NEW_MASTER`.


In [ ]:
from pathlib import Path

def find_package_root() -> Path:
    current = Path.cwd().resolve()
    candidates = [current, *current.parents]
    for candidate in candidates:
        if (candidate / "01_DATA_PIPELINE").is_dir() and (candidate / "02_MODELS").is_dir():
            return candidate
    if current.name == "01_DATA_PIPELINE":
        return current.parent
    raise RuntimeError("Open this notebook from inside the SUBMIT_THIS package.")

PACKAGE_ROOT = find_package_root()
PIPELINE_DIR = PACKAGE_ROOT / "01_DATA_PIPELINE"
FROZEN_RAW_DIR = PIPELINE_DIR / "FROZEN_RAW_DATA"
GENERATED_DIR = PIPELINE_DIR / "GENERATED_OUTPUTS"
GENERATED_DIR.mkdir(parents=True, exist_ok=True)
print("Package root:", PACKAGE_ROOT)


In [ ]:
import os
import importlib.util
import shutil
import subprocess
import sys

required = {"joblib": "joblib", "sklearn": "scikit-learn", "pandas": "pandas", "numpy": "numpy"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])

DATA_SOURCE = "frozen"  # Change to "live" only after notebook 01 finishes successfully.
if DATA_SOURCE == "frozen":
    RAW_PATH = FROZEN_RAW_DIR / "quarterly_fundamental_raw.csv"
else:
    RAW_PATH = GENERATED_DIR / "01_RAW_LIVE" / "data" / "quarterly_fundamental_raw.csv"

OUTPUT_DIR = GENERATED_DIR / "03_NEW_MASTER"
RUNTIME_DIR = OUTPUT_DIR / "_runtime_source"
PROJECT_LAYERS = OUTPUT_DIR / "rebuild_layers"
EXPORT_DIR = OUTPUT_DIR / "exports"
FROZEN_MASTER = PACKAGE_ROOT / "02_MODELS" / "04_ENS035" / "common_master.csv"

if not RAW_PATH.is_file():
    raise FileNotFoundError(RAW_PATH)
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
RUNTIME_DIR.mkdir(parents=True)
EXPORT_DIR.mkdir(parents=True)
print("Raw input:", RAW_PATH)
print("Output:", OUTPUT_DIR)


In [ ]:
EMBEDDED_SOURCES = {'build_earnings_risk_release.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport importlib\nimport json\nimport sys\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n\nTARGET = "target_earnings_deterioration_10pct"\nCONTINUOUS_TARGET = "next_quarter_net_income_yoy"\nEXPECTED = {\n    "raw_rows": 1458,\n    "labeled_rows": 1202,\n    "train_rows": 968,\n    "test_rows": 234,\n    "test_risk": 40,\n    "test_norisk": 194,\n    "source_features": 258,\n    "reportnorm_features": 309,\n    "union_features": 357,\n}\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser(\n        description="Build the frozen ENS035/PQENS087 earnings-risk release from raw quarterly data."\n    )\n    parser.add_argument("--raw", required=True, type=Path)\n    parser.add_argument("--canonical-source-dir", required=True, type=Path)\n    parser.add_argument("--project-data-root", required=True, type=Path)\n    parser.add_argument("--bundle-export-dir", required=True, type=Path)\n    return parser.parse_args()\n\n\ndef sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef write_json(path: Path, value: object) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding="utf-8")\n\n\ndef clean_raw(raw: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n    clean = raw.copy()\n    clean["ticker"] = clean["ticker"].astype("string").str.strip().str.upper()\n    clean["report_year"] = pd.to_numeric(clean["report_year"], errors="coerce").astype("Int64")\n    clean["report_quarter"] = pd.to_numeric(clean["report_quarter"], errors="coerce").astype("Int64")\n\n    numeric_columns = [\n        column\n        for column in clean.columns\n        if column.endswith("_million_vnd") or column == "roe_pct"\n    ]\n    for column in numeric_columns:\n        clean[column] = pd.to_numeric(clean[column], errors="coerce")\n\n    core_denominators = [\n        "total_assets_million_vnd",\n        "total_liabilities_and_equity_million_vnd",\n        "customer_loans_million_vnd",\n        "customer_deposits_million_vnd",\n        "equity_million_vnd",\n    ]\n    invalidated_nonpositive: dict[str, int] = {}\n    for column in core_denominators:\n        invalid_mask = clean[column].le(0).fillna(False)\n        invalidated_nonpositive[column] = int(invalid_mask.sum())\n        clean.loc[invalid_mask, column] = np.nan\n\n    for column in ["period_begin_date", "period_end_date"]:\n        if column in clean:\n            parsed = pd.to_datetime(clean[column], errors="coerce")\n            clean[column] = parsed.dt.strftime("%Y-%m-%d")\n\n    # Reserved only. These fields are deliberately excluded from every release feature list.\n    clean["announcement_date"] = pd.NA\n    clean["announcement_date_source"] = pd.NA\n    clean["announcement_date_verified"] = pd.NA\n    clean = clean.sort_values(["ticker", "report_year", "report_quarter"], kind="stable").reset_index(drop=True)\n\n    duplicate_mask = clean.duplicated(["ticker", "report_year", "report_quarter"], keep=False)\n    invalid_quarter = ~clean["report_quarter"].isin([1, 2, 3, 4])\n    invalid_year = clean["report_year"].isna() | clean["report_year"].lt(1990) | clean["report_year"].gt(2100)\n    infinity_count = int(np.isinf(clean[numeric_columns].to_numpy(dtype=float)).sum())\n    nonpositive_assets = int((clean["total_assets_million_vnd"] <= 0).fillna(False).sum())\n\n    checks = [\n        ("raw_row_count", len(clean), EXPECTED["raw_rows"], len(clean) == EXPECTED["raw_rows"]),\n        ("duplicate_ticker_quarter", int(duplicate_mask.sum()), 0, not duplicate_mask.any()),\n        ("invalid_quarter", int(invalid_quarter.sum()), 0, not invalid_quarter.any()),\n        ("invalid_year", int(invalid_year.sum()), 0, not invalid_year.any()),\n        ("numeric_infinity", infinity_count, 0, infinity_count == 0),\n        ("nonpositive_total_assets", nonpositive_assets, 0, nonpositive_assets == 0),\n    ]\n    quality = pd.DataFrame(checks, columns=["check", "actual", "expected", "passed"])\n    for column, count in invalidated_nonpositive.items():\n        quality.loc[len(quality)] = [\n            f"invalidated_nonpositive::{column}",\n            count,\n            "set_to_missing",\n            True,\n        ]\n    for column in clean.columns:\n        quality.loc[len(quality)] = [\n            f"missing::{column}",\n            int(clean[column].isna().sum()),\n            "informational",\n            True,\n        ]\n    if not quality.loc[~quality.check.str.startswith("missing::"), "passed"].all():\n        failures = quality.loc[~quality.passed, ["check", "actual", "expected"]]\n        raise ValueError(f"Raw data quality gate failed:\\n{failures.to_string(index=False)}")\n    return clean, quality\n\n\ndef import_canonical(source_dir: Path):\n    source_dir = source_dir.resolve()\n    required = [\n        "run_clean_financial_deterioration.py",\n        "run_source_guided_experiments.py",\n        "run_reportnorm_expansion.py",\n    ]\n    missing = [name for name in required if not (source_dir / name).is_file()]\n    if missing:\n        raise FileNotFoundError(f"Canonical source files missing: {missing}")\n    sys.path.insert(0, str(source_dir))\n    clean_module = importlib.import_module("run_clean_financial_deterioration")\n    source_module = importlib.import_module("run_source_guided_experiments")\n    reportnorm_module = importlib.import_module("run_reportnorm_expansion")\n    return clean_module, source_module, reportnorm_module\n\n\ndef no_future_columns(features: list[str]) -> None:\n    forbidden_prefixes = ("next_quarter_", "target_")\n    forbidden_exact = {\n        "target_year",\n        "target_quarter",\n        "target_quarter_number",\n        "announcement_date",\n        "announcement_date_source",\n        "announcement_date_verified",\n    }\n    leakage = [\n        column\n        for column in features\n        if column.startswith(forbidden_prefixes) or column in forbidden_exact\n    ]\n    if leakage:\n        raise ValueError(f"Feature leakage detected: {leakage}")\n\n\ndef feature_list_frame(features: list[str], feature_set: str) -> pd.DataFrame:\n    return pd.DataFrame(\n        {\n            "feature_order": np.arange(1, len(features) + 1),\n            "feature_name": features,\n            "feature_set": feature_set,\n            "rapidminer_role": "regular",\n        }\n    )\n\n\ndef label_text(values: pd.Series) -> pd.Series:\n    return values.astype(int).map({0: "NoRisk", 1: "Risk"})\n\n\ndef metadata_columns() -> list[str]:\n    return [\n        "row_key",\n        "ticker",\n        "report_year",\n        "report_quarter",\n        "quarter_index",\n        "feature_quarter",\n        "target_year",\n        "target_quarter_number",\n        "target_q_index",\n        "target_quarter",\n        "risk10_numeric",\n        "risk10_label",\n        "aux_risk5_numeric",\n        "aux_risk5_label",\n        "mg002_distance_weight",\n        "mg035_positive_weight",\n    ]\n\n\ndef build_prequential(\n    labeled: pd.DataFrame,\n    union_features: list[str],\n    target_indices: list[int],\n    expected_score_rows: int,\n) -> pd.DataFrame:\n    parts: list[pd.DataFrame] = []\n    keep = metadata_columns() + union_features\n    for target_index in sorted(target_indices):\n        target_index = int(target_index)\n        evaluation = labeled[labeled.target_q_index == target_index].copy()\n        train = labeled[\n            (labeled.target_q_index < target_index)\n            & (labeled.target_q_index >= target_index - 40)\n        ].copy()\n        age_years = (target_index - train.target_q_index) / 4.0\n        train["pq_sample_weight"] = np.power(0.95, age_years)\n        train["pq_age_years"] = age_years\n        train["pq_role"] = "train"\n        evaluation["pq_sample_weight"] = 1.0\n        evaluation["pq_age_years"] = 0.0\n        evaluation["pq_role"] = "score"\n        evaluation_quarter = str(evaluation.target_quarter.iloc[0])\n        for frame in (train, evaluation):\n            frame["evaluation_target_q_index"] = target_index\n            frame["evaluation_target_quarter"] = evaluation_quarter\n            parts.append(\n                frame[\n                    [\n                        "evaluation_target_q_index",\n                        "evaluation_target_quarter",\n                        "pq_role",\n                        "pq_sample_weight",\n                        "pq_age_years",\n                    ]\n                    + keep\n                ]\n            )\n    result = pd.concat(parts, ignore_index=True)\n    score = result[result.pq_role == "score"]\n    if len(score) != expected_score_rows or score.row_key.duplicated().any():\n        raise ValueError("PQ prequential export must contain one score row per evaluation example")\n    return result\n\n\ndef data_dictionary(frame: pd.DataFrame, source_features: set[str], reportnorm_features: set[str]) -> pd.DataFrame:\n    rows = []\n    for column in frame.columns:\n        if column in source_features and column in reportnorm_features:\n            role = "regular_feature_shared"\n        elif column in source_features:\n            role = "regular_feature_258"\n        elif column in reportnorm_features:\n            role = "regular_feature_309"\n        elif column.endswith("_label"):\n            role = "label"\n        elif column.endswith("_weight"):\n            role = "weight"\n        elif column in {"row_key", "ticker", "feature_quarter", "target_quarter"}:\n            role = "id_or_metadata"\n        else:\n            role = "metadata"\n        rows.append(\n            {\n                "column_name": column,\n                "pandas_dtype": str(frame[column].dtype),\n                "role": role,\n                "missing_count": int(frame[column].isna().sum()),\n                "description": "Generated by the frozen 2026-06-19 earnings-risk release pipeline.",\n            }\n        )\n    return pd.DataFrame(rows)\n\n\ndef write_csv(frame: pd.DataFrame, project_path: Path, bundle_path: Path) -> None:\n    project_path.parent.mkdir(parents=True, exist_ok=True)\n    bundle_path.parent.mkdir(parents=True, exist_ok=True)\n    frame.to_csv(project_path, index=False, encoding="utf-8-sig")\n    frame.to_csv(bundle_path, index=False, encoding="utf-8-sig")\n\n\ndef main() -> None:\n    args = parse_args()\n    raw_path = args.raw.resolve()\n    project = args.project_data_root.resolve()\n    export = args.bundle_export_dir.resolve()\n    for path in [raw_path, args.canonical_source_dir]:\n        if not path.exists():\n            raise FileNotFoundError(path)\n\n    raw = pd.read_csv(raw_path, low_memory=False)\n    clean, quality = clean_raw(raw)\n    clean_module, source_module, reportnorm_module = import_canonical(args.canonical_source_dir)\n\n    clean_dir = project / "1.2 chuẩn hóa và làm sạch data"\n    sync_dir = project / "1.3 đồng bộ hóa data theo quý"\n    feature_dir = project / "1.4 tạo feature"\n    label_dir = project / "1.5 tạo label"\n    final_dir = project / "1.6 dataset cuối cùng để đưa vào rapidminer"\n\n    for stage_dir in [clean_dir, sync_dir, feature_dir, label_dir, final_dir]:\n        stage_dir.mkdir(parents=True, exist_ok=True)\n\n    clean_path = clean_dir / "earnings_quarterly_clean.csv"\n    clean.to_csv(clean_path, index=False, encoding="utf-8-sig")\n    quality.to_csv(clean_dir / "data_quality_report.csv", index=False, encoding="utf-8-sig")\n    source_manifest = {\n        "raw_path": str(raw_path),\n        "raw_sha256": sha256(raw_path),\n        "raw_rows": len(raw),\n        "clean_path": str(clean_path),\n        "clean_sha256": sha256(clean_path),\n        "announcement_date_release_status": "schema_reserved_not_used",\n    }\n    write_json(clean_dir / "source_checksum.json", source_manifest)\n\n    audit, base_features, _ = clean_module.build_dataset(clean)\n    audit["target_q_index"] = (\n        pd.to_numeric(audit.target_year, errors="coerce") * 4\n        + pd.to_numeric(audit.target_quarter_number, errors="coerce")\n        - 1\n    ).astype("Int64")\n    audit["row_key"] = (\n        audit.ticker.astype(str)\n        + "__"\n        + audit.feature_quarter.astype(str)\n        + "__"\n        + audit.target_quarter.astype(str)\n    )\n    audit.to_csv(sync_dir / "earnings_quarter_sequence.csv", index=False, encoding="utf-8-sig")\n\n    source_frame, feature_map, _ = source_module.build_source_guided_features(audit, base_features)\n    expanded, reportnorm_map, _ = reportnorm_module.add_reportnorm_features(source_frame, feature_map)\n    source_features = feature_map["source_full_ticker"]\n    reportnorm_features = reportnorm_map["reportnorm_ticker"]\n    union_features = list(dict.fromkeys(source_features + reportnorm_features))\n    no_future_columns(union_features)\n    actual_feature_counts = {\n        "source_features": len(source_features),\n        "reportnorm_features": len(reportnorm_features),\n        "union_features": len(union_features),\n    }\n    for key, actual in actual_feature_counts.items():\n        if actual != EXPECTED[key]:\n            raise ValueError(f"{key}: expected {EXPECTED[key]}, got {actual}")\n\n    expanded[union_features] = expanded[union_features].replace([np.inf, -np.inf], np.nan)\n    expanded["row_key"] = audit["row_key"].to_numpy()\n    expanded["target_q_index"] = audit["target_q_index"].to_numpy()\n    expanded["aux_risk5_numeric"] = np.where(\n        expanded[CONTINUOUS_TARGET].notna(),\n        (expanded[CONTINUOUS_TARGET] <= -0.05).astype(int),\n        np.nan,\n    )\n    labeled = expanded[expanded[TARGET].notna()].copy()\n    labeled["risk10_numeric"] = labeled[TARGET].astype(int)\n    labeled["risk10_label"] = label_text(labeled["risk10_numeric"])\n    labeled["aux_risk5_numeric"] = labeled["aux_risk5_numeric"].astype(int)\n    labeled["aux_risk5_label"] = label_text(labeled["aux_risk5_numeric"])\n    distance = (labeled[CONTINUOUS_TARGET] + 0.05).abs()\n    labeled["mg002_distance_weight"] = 0.20 + 0.80 * np.clip(distance / 0.30, 0, 1)\n    labeled["mg035_positive_weight"] = np.where(labeled.aux_risk5_numeric == 1, 1.5, 1.0)\n\n    meta = metadata_columns()\n    common = labeled[meta + union_features].copy()\n    train = common[labeled.target_year < 2024].copy()\n    validation = common[labeled.target_year.between(2020, 2023)].copy()\n    test = common[labeled.target_year >= 2024].copy()\n    checks = {\n        "labeled_rows": len(common),\n        "train_rows": len(train),\n        "test_rows": len(test),\n        "test_risk": int((test.risk10_label == "Risk").sum()),\n        "test_norisk": int((test.risk10_label == "NoRisk").sum()),\n    }\n    for key, actual in checks.items():\n        if actual != EXPECTED[key]:\n            raise ValueError(f"{key}: expected {EXPECTED[key]}, got {actual}")\n    if common.row_key.duplicated().any():\n        raise ValueError("Duplicate row_key in common master")\n    if np.isinf(common[union_features].to_numpy(dtype=float)).any():\n        raise ValueError("Infinity remains in regular features")\n\n    source_output = expanded[["row_key"] + source_features].copy()\n    reportnorm_output = expanded[["row_key"] + reportnorm_features].copy()\n    source_output.to_csv(feature_dir / "earnings_source_features.csv", index=False, encoding="utf-8-sig")\n    reportnorm_output.to_csv(feature_dir / "earnings_reportnorm_features.csv", index=False, encoding="utf-8-sig")\n    feature_list_frame(source_features, "source_full_ticker").to_csv(\n        feature_dir / "feature_list_258.csv", index=False, encoding="utf-8-sig"\n    )\n    feature_list_frame(reportnorm_features, "reportnorm_ticker").to_csv(\n        feature_dir / "feature_list_309.csv", index=False, encoding="utf-8-sig"\n    )\n\n    label_columns = [\n        "row_key",\n        "ticker",\n        "feature_quarter",\n        "target_quarter",\n        "quarter_index",\n        "target_q_index",\n        "net_income_million_vnd",\n        "next_quarter_net_income",\n        CONTINUOUS_TARGET,\n        "risk10_numeric",\n        "risk10_label",\n        "aux_risk5_numeric",\n        "aux_risk5_label",\n        "mg002_distance_weight",\n        "mg035_positive_weight",\n    ]\n    label_audit = labeled[label_columns].copy()\n    label_audit.to_csv(label_dir / "earnings_risk_labels.csv", index=False, encoding="utf-8-sig")\n\n    pq_test_indices = labeled.loc[labeled.target_year >= 2024, "target_q_index"].unique().tolist()\n    pq_validation_indices = labeled.loc[\n        labeled.target_year.between(2020, 2023), "target_q_index"\n    ].unique().tolist()\n    pq = build_prequential(labeled, union_features, pq_test_indices, EXPECTED["test_rows"])\n    pq_validation = build_prequential(\n        labeled, union_features, pq_validation_indices, len(validation)\n    )\n    dictionary = data_dictionary(common, set(source_features), set(reportnorm_features))\n    exports = {\n        "common_master.csv": common,\n        "ens035_train.csv": train,\n        "ens035_validation.csv": validation,\n        "ens035_test.csv": test,\n        "pq087_prequential.csv": pq,\n        "pq087_validation_prequential.csv": pq_validation,\n        "feature_list_258.csv": feature_list_frame(source_features, "source_full_ticker"),\n        "feature_list_309.csv": feature_list_frame(reportnorm_features, "reportnorm_ticker"),\n        "data_dictionary.csv": dictionary,\n    }\n    for name, frame in exports.items():\n        write_csv(frame, final_dir / name, export / name)\n\n    release_checks = pd.DataFrame(\n        [\n            {"check": key, "actual": value, "expected": EXPECTED[key], "passed": value == EXPECTED[key]}\n            for key, value in {**actual_feature_counts, **checks, "raw_rows": len(raw)}.items()\n        ]\n        + [\n            {"check": "duplicate_row_key", "actual": int(common.row_key.duplicated().sum()), "expected": 0, "passed": True},\n            {"check": "feature_infinity", "actual": 0, "expected": 0, "passed": True},\n            {"check": "feature_leakage", "actual": 0, "expected": 0, "passed": True},\n            {"check": "pq_score_rows", "actual": int((pq.pq_role == "score").sum()), "expected": EXPECTED["test_rows"], "passed": int((pq.pq_role == "score").sum()) == EXPECTED["test_rows"]},\n        ]\n    )\n    release_checks.to_csv(final_dir / "ACCEPTANCE_CHECKS.csv", index=False, encoding="utf-8-sig")\n    release_checks.to_csv(export / "ACCEPTANCE_CHECKS.csv", index=False, encoding="utf-8-sig")\n    manifest = {\n        "target": TARGET,\n        "target_definition": "Risk when immediately next-quarter net-income YoY change is <= -10%.",\n        "auxiliary_training_target": "Risk when immediately next-quarter net-income YoY change is <= -5%.",\n        "raw_source": source_manifest,\n        "feature_counts": actual_feature_counts,\n        "row_counts": checks,\n        "feature_overlap": len(set(source_features) & set(reportnorm_features)),\n        "announcement_date_used": False,\n        "exports": {name: {"rows": len(frame), "columns": len(frame.columns)} for name, frame in exports.items()},\n    }\n    write_json(final_dir / "DATASET_MANIFEST.json", manifest)\n    write_json(export / "DATASET_MANIFEST.json", manifest)\n    print(json.dumps(manifest, indent=2, ensure_ascii=True))\n\n\nif __name__ == "__main__":\n    main()\n', 'run_clean_financial_deterioration.py': 'from __future__ import annotations\n\nimport argparse\nimport json\nimport time\nimport warnings\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.ensemble import (\n    ExtraTreesClassifier,\n    GradientBoostingClassifier,\n    HistGradientBoostingClassifier,\n    RandomForestClassifier,\n)\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import RobustScaler\nfrom sklearn.tree import DecisionTreeClassifier\n\nwarnings.filterwarnings("ignore")\nSEED = 20260618\nVALIDATION_YEARS = (2020, 2021, 2022, 2023)\nTARGET = "target_earnings_deterioration_10pct"\n\n\n@dataclass(frozen=True)\nclass Candidate:\n    candidate_id: str\n    family: str\n    feature_mode: str\n    params: dict\n\n\ndef metric_row(y: np.ndarray, pred: np.ndarray, prob: np.ndarray) -> dict:\n    y = np.asarray(y).astype(int)\n    pred = np.asarray(pred).astype(int)\n    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()\n    return {\n        "accuracy": accuracy_score(y, pred),\n        "balanced_accuracy": balanced_accuracy_score(y, pred),\n        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),\n        "risk_precision": precision_score(y, pred, pos_label=1, zero_division=0),\n        "risk_recall": recall_score(y, pred, pos_label=1, zero_division=0),\n        "norisk_precision": precision_score(y, pred, pos_label=0, zero_division=0),\n        "norisk_recall": recall_score(y, pred, pos_label=0, zero_division=0),\n        "pr_auc": average_precision_score(y, prob) if len(np.unique(y)) == 2 else np.nan,\n        "roc_auc": roc_auc_score(y, prob) if len(np.unique(y)) == 2 else np.nan,\n        "predicted_risk_rate": float(pred.mean()),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n\n\ndef safe_div(a: pd.Series, b: pd.Series) -> pd.Series:\n    return (a / b.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)\n\n\ndef build_dataset(raw: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:\n    q = raw.sort_values(["ticker", "report_year", "report_quarter"]).copy()\n    q["quarter_index"] = q.report_year.astype(int) * 4 + q.report_quarter.astype(int) - 1\n    q["feature_quarter"] = q.report_year.astype(str) + "Q" + q.report_quarter.astype(str)\n\n    base = [\n        "roe_pct",\n        "operating_profit_before_provision_million_vnd",\n        "total_assets_million_vnd",\n        "customer_loans_million_vnd",\n        "customer_deposits_million_vnd",\n        "equity_million_vnd",\n        "net_income_million_vnd",\n        "net_interest_income_million_vnd",\n        "provision_expense_million_vnd",\n    ]\n    for col in base:\n        q[col] = pd.to_numeric(q[col], errors="coerce")\n\n    q["loan_to_assets"] = safe_div(q.customer_loans_million_vnd, q.total_assets_million_vnd)\n    q["deposits_to_assets"] = safe_div(q.customer_deposits_million_vnd, q.total_assets_million_vnd)\n    q["equity_to_assets"] = safe_div(q.equity_million_vnd, q.total_assets_million_vnd)\n    q["loan_to_deposit"] = safe_div(q.customer_loans_million_vnd, q.customer_deposits_million_vnd)\n    q["net_interest_to_assets"] = safe_div(q.net_interest_income_million_vnd, q.total_assets_million_vnd)\n    q["net_income_to_equity"] = safe_div(q.net_income_million_vnd, q.equity_million_vnd)\n    q["provision_to_preprovision_profit"] = safe_div(\n        q.provision_expense_million_vnd.abs(),\n        q.operating_profit_before_provision_million_vnd.abs(),\n    )\n\n    flow_cols = [\n        "roe_pct",\n        "operating_profit_before_provision_million_vnd",\n        "total_assets_million_vnd",\n        "customer_loans_million_vnd",\n        "customer_deposits_million_vnd",\n        "equity_million_vnd",\n        "net_income_million_vnd",\n        "net_interest_income_million_vnd",\n        "provision_expense_million_vnd",\n        "loan_to_assets",\n        "deposits_to_assets",\n        "equity_to_assets",\n        "loan_to_deposit",\n        "net_interest_to_assets",\n        "net_income_to_equity",\n        "provision_to_preprovision_profit",\n    ]\n    for col in flow_cols:\n        previous = q.groupby("ticker")[col].shift(1)\n        year_ago = q.groupby("ticker")[col].shift(4)\n        prev_index = q.groupby("ticker").quarter_index.shift(1)\n        year_index = q.groupby("ticker").quarter_index.shift(4)\n        q[col + "__qoq_change"] = (q[col] - previous).where(q.quarter_index - prev_index == 1)\n        q[col + "__yoy_change"] = (q[col] - year_ago).where(q.quarter_index - year_index == 4)\n        if col.endswith("_million_vnd"):\n            q[col + "__qoq_pct"] = safe_div(q[col] - previous, previous.abs()).where(q.quarter_index - prev_index == 1)\n            q[col + "__yoy_pct"] = safe_div(q[col] - year_ago, year_ago.abs()).where(q.quarter_index - year_index == 4)\n\n    q["loan_growth_yoy"] = q["customer_loans_million_vnd__yoy_pct"]\n    q["deposit_growth_yoy"] = q["customer_deposits_million_vnd__yoy_pct"]\n    q["credit_funding_growth_gap"] = q.loan_growth_yoy - q.deposit_growth_yoy\n\n    next_values = q[\n        [\n            "ticker",\n            "quarter_index",\n            "report_year",\n            "report_quarter",\n            "net_income_million_vnd",\n            "roe_pct",\n            "provision_to_preprovision_profit",\n            "credit_funding_growth_gap",\n        ]\n    ].copy()\n    next_values["quarter_index"] -= 1\n    next_values = next_values.rename(\n        columns={\n            "report_year": "target_year",\n            "report_quarter": "target_quarter_number",\n            "net_income_million_vnd": "next_quarter_net_income",\n            "roe_pct": "next_quarter_roe",\n            "provision_to_preprovision_profit": "next_quarter_provision_ratio",\n            "credit_funding_growth_gap": "next_quarter_credit_funding_gap",\n        }\n    )\n    q = q.merge(next_values, on=["ticker", "quarter_index"], how="left")\n\n    prior_year_income = q.groupby("ticker").net_income_million_vnd.shift(3)\n    prior_index = q.groupby("ticker").quarter_index.shift(3)\n    prior_year_income = prior_year_income.where(q.quarter_index - prior_index == 3)\n    q["next_quarter_net_income_yoy"] = safe_div(\n        q.next_quarter_net_income - prior_year_income,\n        prior_year_income.abs(),\n    )\n    q[TARGET] = np.where(\n        q.next_quarter_net_income.notna() & prior_year_income.notna(),\n        (q.next_quarter_net_income_yoy <= -0.10).astype(float),\n        np.nan,\n    )\n    q["target_earnings_deterioration_20pct"] = np.where(\n        q.next_quarter_net_income.notna() & prior_year_income.notna(),\n        (q.next_quarter_net_income_yoy <= -0.20).astype(float),\n        np.nan,\n    )\n    q["target_roe_drop_2pp"] = np.where(\n        q.next_quarter_roe.notna() & q.roe_pct.notna(),\n        (q.next_quarter_roe <= q.roe_pct - 2.0).astype(float),\n        np.nan,\n    )\n    q["target_provision_ratio_increase_10pp"] = np.where(\n        q.next_quarter_provision_ratio.notna() & q.provision_to_preprovision_profit.notna(),\n        (q.next_quarter_provision_ratio - q.provision_to_preprovision_profit >= 0.10).astype(float),\n        np.nan,\n    )\n    q["target_funding_gap_over_10pp"] = np.where(\n        q.next_quarter_credit_funding_gap.notna(),\n        (q.next_quarter_credit_funding_gap >= 0.10).astype(float),\n        np.nan,\n    )\n    q["target_quarter"] = q.target_year.astype("Int64").astype(str) + "Q" + q.target_quarter_number.astype("Int64").astype(str)\n\n    raw_level_features = base + [\n        "loan_to_assets",\n        "deposits_to_assets",\n        "equity_to_assets",\n        "loan_to_deposit",\n        "net_interest_to_assets",\n        "net_income_to_equity",\n        "provision_to_preprovision_profit",\n        "loan_growth_yoy",\n        "deposit_growth_yoy",\n        "credit_funding_growth_gap",\n    ]\n    trend_features = [\n        c for c in q.columns\n        if c.endswith(("__qoq_change", "__yoy_change", "__qoq_pct", "__yoy_pct"))\n    ]\n    all_features = raw_level_features + trend_features\n    ratio_trend_features = [\n        c for c in all_features\n        if not c.endswith("_million_vnd")\n        and not (c.endswith("_million_vnd__qoq_change") or c.endswith("_million_vnd__yoy_change"))\n    ]\n    return q, sorted(set(all_features)), sorted(set(ratio_trend_features))\n\n\ndef candidates() -> list[Candidate]:\n    rows = []\n    counter = 0\n    for mode in ["all_clean", "ratio_trends"]:\n        for c in [0.005, 0.01, 0.03, 0.1, 0.3, 1.0]:\n            for weight in ["balanced", {0: 1.0, 1: 1.5}]:\n                counter += 1\n                rows.append(Candidate(f"C{counter:03d}", "logreg", mode, {"C": c, "class_weight": weight}))\n        for depth in [4, 7]:\n            for leaf in [8, 16]:\n                counter += 1\n                rows.append(Candidate(f"C{counter:03d}", "extra", mode, {"max_depth": depth, "min_samples_leaf": leaf}))\n        for leaves in [5, 9]:\n            for leaf in [12, 24]:\n                counter += 1\n                rows.append(Candidate(f"C{counter:03d}", "hgb", mode, {"max_leaf_nodes": leaves, "min_samples_leaf": leaf}))\n        for depth in [4, 7]:\n            counter += 1\n            rows.append(Candidate(f"C{counter:03d}", "rf", mode, {"max_depth": depth, "min_samples_leaf": 10}))\n        for depth in [1, 2]:\n            counter += 1\n            rows.append(Candidate(f"C{counter:03d}", "gb", mode, {"max_depth": depth, "min_samples_leaf": 15}))\n        for depth in [2, 3]:\n            counter += 1\n            rows.append(Candidate(f"C{counter:03d}", "tree", mode, {"max_depth": depth, "min_samples_leaf": 15}))\n    return rows\n\n\ndef make_model(candidate: Candidate) -> Pipeline:\n    p = candidate.params\n    if candidate.family == "logreg":\n        model = LogisticRegression(C=p["C"], class_weight=p["class_weight"], max_iter=5000, random_state=SEED)\n        scale = RobustScaler()\n    elif candidate.family == "extra":\n        model = ExtraTreesClassifier(n_estimators=400, max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"], max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=SEED)\n        scale = "passthrough"\n    elif candidate.family == "hgb":\n        model = HistGradientBoostingClassifier(max_iter=300, learning_rate=.04, max_leaf_nodes=p["max_leaf_nodes"], min_samples_leaf=p["min_samples_leaf"], l2_regularization=.1, random_state=SEED)\n        scale = "passthrough"\n    elif candidate.family == "rf":\n        model = RandomForestClassifier(n_estimators=400, max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"], max_features="sqrt", class_weight="balanced", n_jobs=-1, random_state=SEED)\n        scale = "passthrough"\n    elif candidate.family == "gb":\n        model = GradientBoostingClassifier(n_estimators=250, learning_rate=.04, max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"], subsample=.8, random_state=SEED)\n        scale = "passthrough"\n    else:\n        model = DecisionTreeClassifier(max_depth=p["max_depth"], min_samples_leaf=p["min_samples_leaf"], class_weight="balanced", random_state=SEED)\n        scale = "passthrough"\n    return Pipeline([\n        ("imputer", SimpleImputer(strategy="median", add_indicator=True)),\n        ("scale", scale),\n        ("model", model),\n    ])\n\n\ndef fit_oof(frame: pd.DataFrame, features: list[str], candidate: Candidate) -> pd.DataFrame:\n    parts = []\n    for year in VALIDATION_YEARS:\n        train = frame[(frame.target_year < year) & frame[TARGET].notna()].copy()\n        validation = frame[(frame.target_year == year) & frame[TARGET].notna()].copy()\n        if len(train) < 150 or len(validation) < 20 or train[TARGET].nunique() < 2 or validation[TARGET].nunique() < 2:\n            continue\n        model = make_model(candidate)\n        model.fit(train[features], train[TARGET].astype(int))\n        prob = model.predict_proba(validation[features])[:, 1]\n        parts.append(pd.DataFrame({\n            "row_index": validation.index,\n            "fold": year,\n            "ticker": validation.ticker.to_numpy(),\n            "feature_quarter": validation.feature_quarter.to_numpy(),\n            "target_quarter": validation.target_quarter.to_numpy(),\n            "y": validation[TARGET].astype(int).to_numpy(),\n            "probability": prob,\n        }))\n    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()\n\n\ndef choose_threshold(oof: pd.DataFrame) -> tuple[pd.Series, pd.DataFrame]:\n    rows = []\n    for threshold in np.arange(.05, .951, .01):\n        folds = []\n        for fold, group in oof.groupby("fold"):\n            folds.append({"fold": fold, **metric_row(group.y, group.probability >= threshold, group.probability)})\n        f = pd.DataFrame(folds)\n        rows.append({\n            "threshold": threshold,\n            "mean_accuracy": f.accuracy.mean(),\n            "mean_macro_f1": f.macro_f1.mean(),\n            "mean_risk_precision": f.risk_precision.mean(),\n            "mean_risk_recall": f.risk_recall.mean(),\n            "mean_norisk_precision": f.norisk_precision.mean(),\n            "mean_norisk_recall": f.norisk_recall.mean(),\n            "mean_pr_auc": f.pr_auc.mean(),\n            "worst_macro_f1": f.macro_f1.min(),\n            "worst_risk_recall": f.risk_recall.min(),\n            "worst_norisk_recall": f.norisk_recall.min(),\n            "std_macro_f1": f.macro_f1.std(ddof=0),\n        })\n    grid = pd.DataFrame(rows)\n    grid["min_mean_class_recall"] = grid[["mean_risk_recall", "mean_norisk_recall"]].min(axis=1)\n    grid["selection_score"] = (\n        1.5 * grid.min_mean_class_recall\n        + grid.mean_macro_f1\n        + .3 * grid.worst_macro_f1\n        + .2 * grid.mean_pr_auc\n        - .25 * grid.std_macro_f1\n    )\n    best = grid.sort_values(["selection_score", "mean_macro_f1"], ascending=False).iloc[0]\n    return best, grid\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--input", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    args = parser.parse_args()\n    args.output.mkdir(parents=True, exist_ok=True)\n    for name in ["datasets", "results", "models", "notes"]:\n        (args.output / name).mkdir(exist_ok=True)\n    started = time.time()\n\n    raw = pd.read_csv(args.input)\n    audit, all_features, ratio_features = build_dataset(raw)\n    audit.to_csv(args.output / "datasets" / "quarter_sequence_audit.csv", index=False)\n    feature_map = {"all_clean": all_features, "ratio_trends": ratio_features}\n\n    label_cols = [\n        c for c in audit.columns\n        if c.startswith("target_")\n        and c not in {"target_year", "target_quarter", "target_quarter_number"}\n    ]\n    future_cols = [\n        "next_quarter_net_income",\n        "next_quarter_roe",\n        "next_quarter_provision_ratio",\n        "next_quarter_credit_funding_gap",\n        "next_quarter_net_income_yoy",\n    ]\n    ml_ready_cols = ["ticker", "feature_quarter", "target_quarter", "target_year", "target_quarter_number"] + all_features + [TARGET]\n    ml_ready = audit[ml_ready_cols].dropna(subset=[TARGET]).copy()\n    ml_ready.to_csv(args.output / "datasets" / "earnings_deterioration_ml_ready.csv", index=False)\n\n    label_summary = []\n    for col in label_cols:\n        valid = audit[col].dropna()\n        label_summary.append({"target": col, "rows": len(valid), "risk_count": int(valid.sum()), "risk_rate": valid.mean()})\n    pd.DataFrame(label_summary).to_csv(args.output / "results" / "target_distribution.csv", index=False)\n\n    candidate_results = []\n    oof_cache = {}\n    for number, candidate in enumerate(candidates(), 1):\n        features = feature_map[candidate.feature_mode]\n        oof = fit_oof(audit, features, candidate)\n        if oof.empty:\n            continue\n        best, grid = choose_threshold(oof)\n        candidate_results.append({\n            "candidate_id": candidate.candidate_id,\n            "family": candidate.family,\n            "feature_mode": candidate.feature_mode,\n            "params_json": json.dumps(candidate.params),\n            "n_features": len(features),\n            **best.to_dict(),\n        })\n        oof_cache[candidate.candidate_id] = oof\n        if number % 10 == 0:\n            print(f"candidate {number}/{len(candidates())}", flush=True)\n\n    results = pd.DataFrame(candidate_results).sort_values(\n        ["selection_score", "mean_macro_f1"], ascending=False\n    )\n    results.to_csv(args.output / "results" / "all_candidate_walkforward_results.csv", index=False)\n    winner_row = results.iloc[0]\n    winner = next(c for c in candidates() if c.candidate_id == winner_row.candidate_id)\n    winner_oof = oof_cache[winner.candidate_id]\n    winner_oof.to_csv(args.output / "results" / "winner_oof_predictions.csv", index=False)\n    _, threshold_grid = choose_threshold(winner_oof)\n    threshold_grid.to_csv(args.output / "results" / "winner_threshold_grid.csv", index=False)\n\n    features = feature_map[winner.feature_mode]\n    train = audit[(audit.target_year < 2024) & audit[TARGET].notna()].copy()\n    test = audit[(audit.target_year >= 2024) & audit[TARGET].notna()].copy()\n    model = make_model(winner)\n    model.fit(train[features], train[TARGET].astype(int))\n    probability = model.predict_proba(test[features])[:, 1]\n    threshold = float(winner_row.threshold)\n    prediction = (probability >= threshold).astype(int)\n    test_metrics = metric_row(test[TARGET].astype(int), prediction, probability)\n\n    scored = test[[\n        "ticker", "feature_quarter", "target_quarter", "target_year",\n        "net_income_million_vnd", "next_quarter_net_income",\n        "next_quarter_net_income_yoy", TARGET,\n    ]].copy()\n    scored["risk_probability"] = probability\n    scored["prediction"] = prediction\n    scored.to_csv(args.output / "results" / "final_test_predictions.csv", index=False)\n    joblib.dump(model, args.output / "models" / "validation_selected_model.joblib")\n\n    split_summary = []\n    for year in VALIDATION_YEARS:\n        split_summary.append({\n            "validation_target_year": year,\n            "train_rows": int(((audit.target_year < year) & audit[TARGET].notna()).sum()),\n            "validation_rows": int(((audit.target_year == year) & audit[TARGET].notna()).sum()),\n        })\n    split_summary.append({"validation_target_year": "final_2024plus", "train_rows": len(train), "validation_rows": len(test)})\n    pd.DataFrame(split_summary).to_csv(args.output / "results" / "time_split_summary.csv", index=False)\n\n    manifest = {\n        "research_question": "Can quarter-t financial conditions predict next-quarter YoY earnings deterioration?",\n        "target_definition": "Risk if next-quarter net income is at least 10% below the same quarter one year earlier.",\n        "grain": "one ticker-quarter feature row predicting the immediately following quarter",\n        "data_sources": [str(args.input)],\n        "excluded_sources": ["daily market", "company snapshot", "annual OCR", "VNINDEX", "USDVND"],\n        "proxy_dates_used": False,\n        "candidate_count": len(results),\n        "winner": winner.__dict__,\n        "threshold": threshold,\n        "feature_count": len(features),\n        "features": features,\n        "validation_metrics": winner_row.to_dict(),\n        "test_metrics": test_metrics,\n        "runtime_seconds": time.time() - started,\n    }\n    (args.output / "models" / "manifest.json").write_text(json.dumps(manifest, indent=2, default=str), encoding="utf-8")\n\n    passed_joint = test_metrics["accuracy"] >= .60 and test_metrics["risk_recall"] >= .70 and test_metrics["norisk_recall"] >= .70\n    lines = [\n        "# Clean financial deterioration result",\n        "",\n        "## Setup",\n        "",\n        "- One row uses only financial information from quarter t.",\n        "- The label belongs to the immediately following quarter t+1.",\n        "- No disclosure-date proxy, market price, company snapshot, OCR, VNINDEX, or USDVND is used.",\n        "- Model and threshold are selected using target-quarter validation years 2020-2023.",\n        "- Target quarters 2024+ are used once for final confirmation.",\n        "",\n        "## Selected configuration",\n        "",\n        f"- Model: `{winner.family}` / `{winner.candidate_id}`.",\n        f"- Feature mode: `{winner.feature_mode}` ({len(features)} features).",\n        f"- Threshold: `{threshold:.3f}`.",\n        "",\n        "## Final test",\n        "",\n        f"- Rows: {len(test)}.",\n        f"- Accuracy: {test_metrics[\'accuracy\']:.2%}.",\n        f"- Risk precision: {test_metrics[\'risk_precision\']:.2%}.",\n        f"- Risk recall: {test_metrics[\'risk_recall\']:.2%}.",\n        f"- NoRisk precision: {test_metrics[\'norisk_precision\']:.2%}.",\n        f"- NoRisk recall: {test_metrics[\'norisk_recall\']:.2%}.",\n        f"- Macro F1: {test_metrics[\'macro_f1\']:.2%}.",\n        f"- PR-AUC: {test_metrics[\'pr_auc\']:.3f}.",\n        f"- Confusion matrix TN/FP/FN/TP: {test_metrics[\'tn\']}/{test_metrics[\'fp\']}/{test_metrics[\'fn\']}/{test_metrics[\'tp\']}.",\n        "",\n        f"Joint 60/70/70 target passed: **{passed_joint}**.",\n    ]\n    (args.output / "notes" / "FINAL_RESULT.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    print(json.dumps({"winner": winner.candidate_id, "family": winner.family, "threshold": threshold, "test": test_metrics, "joint_pass": passed_joint}, indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'run_source_guided_experiments.py': 'from __future__ import annotations\n\nimport json\nimport math\nimport time\nimport warnings\nfrom dataclasses import asdict, dataclass\nfrom itertools import combinations\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\nfrom sklearn.base import BaseEstimator, TransformerMixin\nfrom sklearn.ensemble import (\n    ExtraTreesClassifier,\n    GradientBoostingClassifier,\n    HistGradientBoostingClassifier,\n    RandomForestClassifier,\n)\nfrom sklearn.impute import SimpleImputer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.metrics import (\n    accuracy_score,\n    average_precision_score,\n    balanced_accuracy_score,\n    confusion_matrix,\n    f1_score,\n    precision_score,\n    recall_score,\n    roc_auc_score,\n)\nfrom sklearn.mixture import GaussianMixture\nfrom sklearn.naive_bayes import GaussianNB\nfrom sklearn.pipeline import Pipeline\nfrom sklearn.preprocessing import RobustScaler, StandardScaler\nfrom sklearn.svm import SVC\n\nwarnings.filterwarnings("ignore")\n\nSEED = 20260618\nTARGET = "target_earnings_deterioration_10pct"\nVALIDATION_YEARS = (2020, 2021, 2022, 2023)\nBASE_DIR = Path(__file__).resolve().parents[1]\nINPUT = BASE_DIR / "output" / "datasets" / "quarter_sequence_audit.csv"\nBASE_MANIFEST = BASE_DIR / "output" / "models" / "manifest.json"\nOUT = Path(__file__).resolve().parent / "output"\n\n\n@dataclass(frozen=True)\nclass Candidate:\n    candidate_id: str\n    family: str\n    feature_mode: str\n    params: dict\n    winsorize: bool = False\n    gmm_components: int = 0\n\n\nclass QuantileClipper(BaseEstimator, TransformerMixin):\n    def __init__(self, lower: float = 0.01, upper: float = 0.99):\n        self.lower = lower\n        self.upper = upper\n\n    def fit(self, x, y=None):\n        values = np.asarray(x, dtype=float)\n        self.lower_bounds_ = np.nanquantile(values, self.lower, axis=0)\n        self.upper_bounds_ = np.nanquantile(values, self.upper, axis=0)\n        return self\n\n    def transform(self, x):\n        return np.clip(np.asarray(x, dtype=float), self.lower_bounds_, self.upper_bounds_)\n\n\ndef metric_row(y, prediction, probability, include_ranking: bool = True) -> dict:\n    y = np.asarray(y).astype(int)\n    prediction = np.asarray(prediction).astype(int)\n    probability = np.asarray(probability, dtype=float)\n    tn, fp, fn, tp = confusion_matrix(y, prediction, labels=[0, 1]).ravel()\n    return {\n        "accuracy": accuracy_score(y, prediction),\n        "balanced_accuracy": balanced_accuracy_score(y, prediction),\n        "macro_f1": f1_score(y, prediction, average="macro", zero_division=0),\n        "risk_precision": precision_score(y, prediction, pos_label=1, zero_division=0),\n        "risk_recall": recall_score(y, prediction, pos_label=1, zero_division=0),\n        "norisk_precision": precision_score(y, prediction, pos_label=0, zero_division=0),\n        "norisk_recall": recall_score(y, prediction, pos_label=0, zero_division=0),\n        "pr_auc": (\n            average_precision_score(y, probability)\n            if include_ranking and len(np.unique(y)) == 2\n            else np.nan\n        ),\n        "roc_auc": (\n            roc_auc_score(y, probability)\n            if include_ranking and len(np.unique(y)) == 2\n            else np.nan\n        ),\n        "predicted_risk_rate": float(prediction.mean()),\n        "tn": int(tn),\n        "fp": int(fp),\n        "fn": int(fn),\n        "tp": int(tp),\n    }\n\n\ndef goal_pass(metrics: dict) -> bool:\n    return (\n        metrics["accuracy"] >= 0.60\n        and metrics["risk_precision"] >= 0.70\n        and metrics["risk_recall"] >= 0.50\n        and metrics["norisk_precision"] >= 0.70\n        and metrics["norisk_recall"] >= 0.50\n    )\n\n\ndef threshold_metrics_fast(y, probability, threshold: float) -> dict:\n    y = np.asarray(y).astype(int)\n    prediction = np.asarray(probability) >= threshold\n    positive = y == 1\n    negative = ~positive\n    tp = int((prediction & positive).sum())\n    fp = int((prediction & negative).sum())\n    fn = int((~prediction & positive).sum())\n    tn = int((~prediction & negative).sum())\n\n    def divide(numerator, denominator):\n        return numerator / denominator if denominator else 0.0\n\n    risk_precision = divide(tp, tp + fp)\n    risk_recall = divide(tp, tp + fn)\n    norisk_precision = divide(tn, tn + fn)\n    norisk_recall = divide(tn, tn + fp)\n    risk_f1 = divide(2 * tp, 2 * tp + fp + fn)\n    norisk_f1 = divide(2 * tn, 2 * tn + fp + fn)\n    return {\n        "accuracy": divide(tp + tn, len(y)),\n        "balanced_accuracy": 0.5 * (risk_recall + norisk_recall),\n        "macro_f1": 0.5 * (risk_f1 + norisk_f1),\n        "risk_precision": risk_precision,\n        "risk_recall": risk_recall,\n        "norisk_precision": norisk_precision,\n        "norisk_recall": norisk_recall,\n        "predicted_risk_rate": float(prediction.mean()),\n        "tn": tn,\n        "fp": fp,\n        "fn": fn,\n        "tp": tp,\n    }\n\n\ndef contiguous_lag(frame: pd.DataFrame, column: str, lag: int) -> pd.Series:\n    grouped = frame.groupby("ticker", sort=False)\n    value = grouped[column].shift(lag)\n    lag_index = grouped.quarter_index.shift(lag)\n    return value.where(frame.quarter_index - lag_index == lag)\n\n\ndef contiguous_rolling(\n    frame: pd.DataFrame,\n    values: pd.Series,\n    window: int,\n    min_periods: int,\n    operation: str,\n    shift: int = 0,\n) -> pd.Series:\n    values = pd.to_numeric(values, errors="coerce")\n    new_segment = frame.groupby("ticker", sort=False)["quarter_index"].diff().ne(1)\n    segment = new_segment.groupby(frame["ticker"], sort=False).cumsum()\n    keys = [frame["ticker"], segment]\n    shifted = values.groupby(keys, sort=False).shift(shift) if shift else values\n\n    def aggregate(series: pd.Series) -> pd.Series:\n        rolling = series.rolling(window, min_periods=min_periods)\n        if operation == "mean":\n            return rolling.mean()\n        if operation == "std":\n            return rolling.std(ddof=0)\n        if operation == "min":\n            return rolling.min()\n        if operation == "max":\n            return rolling.max()\n        if operation == "median":\n            return rolling.median()\n        if operation == "sum":\n            return rolling.sum()\n        raise ValueError(f"Unsupported rolling operation: {operation}")\n\n    return shifted.groupby(keys, sort=False).transform(aggregate)\n\n\ndef rolling_slope(values: np.ndarray) -> float:\n    valid = np.isfinite(values)\n    if valid.sum() < 2:\n        return np.nan\n    x = np.arange(len(values), dtype=float)[valid]\n    y = values[valid].astype(float)\n    return float(np.polyfit(x, y, 1)[0])\n\n\ndef build_source_guided_features(frame: pd.DataFrame, base_features: list[str]):\n    q = frame.sort_values(["ticker", "quarter_index"]).copy()\n    original_index = q.index\n\n    trajectory_sources = [\n        "net_income_million_vnd__yoy_pct",\n        "net_interest_income_million_vnd__yoy_pct",\n        "operating_profit_before_provision_million_vnd__yoy_pct",\n        "provision_expense_million_vnd__yoy_pct",\n        "roe_pct",\n        "net_income_to_equity",\n        "net_interest_to_assets",\n        "provision_to_preprovision_profit",\n        "loan_growth_yoy",\n        "deposit_growth_yoy",\n        "credit_funding_growth_gap",\n        "loan_to_deposit",\n    ]\n    trajectory_sources = [c for c in trajectory_sources if c in q.columns]\n\n    trajectory_features = []\n    outlier_features = []\n    for column in trajectory_sources:\n        short = column.replace("_million_vnd", "")\n        lag1 = contiguous_lag(q, column, 1)\n        lag2 = contiguous_lag(q, column, 2)\n        lag3 = contiguous_lag(q, column, 3)\n        q[f"traj__{short}__lag1"] = lag1\n        q[f"traj__{short}__lag2"] = lag2\n        q[f"traj__{short}__delta1"] = q[column] - lag1\n        q[f"traj__{short}__acceleration"] = q[column] - 2 * lag1 + lag2\n        q[f"traj__{short}__slope3"] = (q[column] - lag2) / 2.0\n        q[f"traj__{short}__slope4"] = (q[column] - lag3) / 3.0\n        q[f"traj__{short}__mean4"] = contiguous_rolling(q, q[column], 4, 2, "mean")\n        q[f"traj__{short}__std4"] = contiguous_rolling(q, q[column], 4, 2, "std")\n        q[f"traj__{short}__range4"] = (\n            contiguous_rolling(q, q[column], 4, 2, "max")\n            - contiguous_rolling(q, q[column], 4, 2, "min")\n        )\n        trajectory_features.extend(\n            [\n                f"traj__{short}__lag1",\n                f"traj__{short}__lag2",\n                f"traj__{short}__delta1",\n                f"traj__{short}__acceleration",\n                f"traj__{short}__slope3",\n                f"traj__{short}__slope4",\n                f"traj__{short}__mean4",\n                f"traj__{short}__std4",\n                f"traj__{short}__range4",\n            ]\n        )\n\n        historical = contiguous_rolling(q, q[column], 1, 1, "mean", shift=1)\n        historical_median = contiguous_rolling(q, q[column], 12, 4, "median", shift=1)\n        historical_mad = contiguous_rolling(\n            q, (historical - historical_median).abs(), 12, 4, "median"\n        )\n        robust_z = (q[column] - historical_median) / (1.4826 * historical_mad.replace(0, np.nan))\n        q[f"outlier__{short}__robust_z"] = robust_z.clip(-20, 20)\n        q[f"outlier__{short}__flag"] = (robust_z.abs() >= 3.0).astype(float)\n        outlier_features.extend(\n            [f"outlier__{short}__robust_z", f"outlier__{short}__flag"]\n        )\n\n    ni_yoy = q["net_income_million_vnd__yoy_pct"]\n    q["traj__current_deterioration_10pct"] = (ni_yoy <= -0.10).astype(float).where(ni_yoy.notna())\n    q["traj__current_deterioration_20pct"] = (ni_yoy <= -0.20).astype(float).where(ni_yoy.notna())\n    q["traj__deterioration_count4"] = contiguous_rolling(\n        q, q["traj__current_deterioration_10pct"], 4, 1, "sum"\n    )\n    q["traj__deterioration_rate8"] = contiguous_rolling(\n        q, q["traj__current_deterioration_10pct"], 8, 2, "mean"\n    )\n    flag = q["traj__current_deterioration_10pct"].fillna(0)\n    flag1 = contiguous_lag(q.assign(_flag=flag), "_flag", 1).fillna(0)\n    flag2 = contiguous_lag(q.assign(_flag=flag), "_flag", 2).fillna(0)\n    flag3 = contiguous_lag(q.assign(_flag=flag), "_flag", 3).fillna(0)\n    q["traj__deterioration_streak"] = flag * (1 + flag1 * (1 + flag2 * (1 + flag3)))\n    trajectory_features.extend(\n        [\n            "traj__current_deterioration_10pct",\n            "traj__current_deterioration_20pct",\n            "traj__deterioration_count4",\n            "traj__deterioration_rate8",\n            "traj__deterioration_streak",\n        ]\n    )\n    q["outlier__flag_count"] = q[[c for c in outlier_features if c.endswith("__flag")]].sum(axis=1)\n    outlier_features.append("outlier__flag_count")\n\n    context_sources = [\n        "net_income_million_vnd__yoy_pct",\n        "net_interest_income_million_vnd__yoy_pct",\n        "operating_profit_before_provision_million_vnd__yoy_pct",\n        "roe_pct",\n        "provision_to_preprovision_profit",\n        "loan_growth_yoy",\n        "deposit_growth_yoy",\n        "credit_funding_growth_gap",\n    ]\n    context_sources = [c for c in context_sources if c in q.columns]\n    context_features = []\n    quarter_group = q.groupby("quarter_index", sort=False)\n    q["ctx__bank_count"] = quarter_group["ticker"].transform("count").astype(float)\n    context_features.append("ctx__bank_count")\n    for column in context_sources:\n        short = column.replace("_million_vnd", "")\n        q[f"ctx__{short}__median"] = quarter_group[column].transform("median")\n        q[f"ctx__{short}__std"] = quarter_group[column].transform("std")\n        q[f"ctx__{short}__negative_rate"] = quarter_group[column].transform(\n            lambda s: float((s < 0).mean())\n        )\n        context_features.extend(\n            [\n                f"ctx__{short}__median",\n                f"ctx__{short}__std",\n                f"ctx__{short}__negative_rate",\n            ]\n        )\n    q["ctx__current_deterioration_rate"] = quarter_group[\n        "traj__current_deterioration_10pct"\n    ].transform("mean")\n    q["ctx__current_severe_deterioration_rate"] = quarter_group[\n        "traj__current_deterioration_20pct"\n    ].transform("mean")\n    context_features.extend(\n        ["ctx__current_deterioration_rate", "ctx__current_severe_deterioration_rate"]\n    )\n\n    ticker_columns = []\n    for ticker in sorted(q.ticker.dropna().unique()):\n        column = f"ticker__{ticker}"\n        q[column] = (q.ticker == ticker).astype(float)\n        ticker_columns.append(column)\n\n    top_base = [\n        "net_income_million_vnd__yoy_pct",\n        "net_income_million_vnd__yoy_change",\n        "net_interest_income_million_vnd__yoy_pct",\n        "net_interest_income_million_vnd__yoy_change",\n        "operating_profit_before_provision_million_vnd__yoy_pct",\n        "operating_profit_before_provision_million_vnd__yoy_change",\n        "provision_expense_million_vnd__yoy_pct",\n        "roe_pct",\n        "roe_pct__yoy_change",\n        "net_income_to_equity",\n        "net_interest_to_assets",\n        "provision_to_preprovision_profit",\n        "loan_growth_yoy",\n        "deposit_growth_yoy",\n        "credit_funding_growth_gap",\n        "loan_to_deposit",\n        "loan_to_deposit__yoy_change",\n        "customer_loans_million_vnd__yoy_pct",\n        "customer_deposits_million_vnd__yoy_pct",\n    ]\n    top_base = [c for c in top_base if c in base_features]\n\n    feature_map = {\n        "base": list(base_features),\n        "trajectory": list(dict.fromkeys(base_features + trajectory_features)),\n        "source_compact": list(\n            dict.fromkeys(top_base + trajectory_features + context_features + outlier_features)\n        ),\n        "source_full": list(\n            dict.fromkeys(base_features + trajectory_features + context_features + outlier_features)\n        ),\n        "source_full_ticker": list(\n            dict.fromkeys(\n                base_features\n                + trajectory_features\n                + context_features\n                + outlier_features\n                + ticker_columns\n            )\n        ),\n    }\n    q = q.loc[original_index]\n    regime_columns = [\n        c\n        for c in context_features\n        if c.endswith(("__median", "__std"))\n        or c in {"ctx__current_deterioration_rate", "ctx__current_severe_deterioration_rate"}\n    ]\n    return q, feature_map, regime_columns\n\n\ndef candidate_specs() -> list[Candidate]:\n    rows = []\n    counter = 0\n\n    def add(family, mode, params, winsor=False, gmm=0):\n        nonlocal counter\n        counter += 1\n        rows.append(Candidate(f"SG{counter:03d}", family, mode, params, winsor, gmm))\n\n    modes = ["base", "trajectory", "source_compact", "source_full", "source_full_ticker"]\n    for mode in modes:\n        rich = mode != "base"\n        for c in ([0.003, 0.01, 0.05, 0.2] if rich else [0.01, 0.05]):\n            for weight in [None, "balanced", {0: 1.0, 1: 1.5}]:\n                add("logreg", mode, {"C": c, "class_weight": weight}, winsor=rich)\n        for depth, leaf, weight in [\n            (4, 8, None),\n            (6, 12, None),\n            (8, 20, None),\n            (4, 10, "balanced"),\n            (7, 16, "balanced"),\n        ]:\n            add("rf", mode, {"max_depth": depth, "min_samples_leaf": leaf, "class_weight": weight})\n            add("extra", mode, {"max_depth": depth, "min_samples_leaf": leaf, "class_weight": weight})\n        for leaves, leaf in [(5, 12), (9, 20), (15, 30)]:\n            add("hgb", mode, {"max_leaf_nodes": leaves, "min_samples_leaf": leaf})\n        for depth, leaf, weight in [(1, 12, 1.0), (1, 20, 1.5), (2, 20, 1.0)]:\n            add("gb", mode, {"max_depth": depth, "min_samples_leaf": leaf, "risk_weight": weight})\n        if mode in {"source_compact", "source_full"}:\n            for c, gamma, weight in [\n                (0.2, "scale", None),\n                (1.0, "scale", None),\n                (3.0, 0.01, None),\n                (1.0, 0.01, "balanced"),\n                (3.0, 0.01, "balanced"),\n            ]:\n                add("svc", mode, {"C": c, "gamma": gamma, "class_weight": weight}, winsor=True)\n            add("gnb", mode, {"var_smoothing": 1e-8}, winsor=True)\n            add("gnb", mode, {"var_smoothing": 1e-6}, winsor=True)\n\n    for mode in ["source_compact", "source_full"]:\n        for components in [2, 3]:\n            add(\n                "logreg",\n                mode,\n                {"C": 0.02, "class_weight": None},\n                winsor=True,\n                gmm=components,\n            )\n            add(\n                "rf",\n                mode,\n                {"max_depth": 6, "min_samples_leaf": 12, "class_weight": None},\n                gmm=components,\n            )\n    return rows\n\n\ndef make_model(candidate: Candidate) -> Pipeline:\n    params = candidate.params\n    if candidate.family == "logreg":\n        estimator = LogisticRegression(\n            C=params["C"],\n            class_weight=params["class_weight"],\n            max_iter=5000,\n            random_state=SEED,\n        )\n        scaler = RobustScaler()\n    elif candidate.family == "rf":\n        estimator = RandomForestClassifier(\n            n_estimators=500,\n            max_depth=params["max_depth"],\n            min_samples_leaf=params["min_samples_leaf"],\n            max_features="sqrt",\n            class_weight=params["class_weight"],\n            n_jobs=-1,\n            random_state=SEED,\n        )\n        scaler = "passthrough"\n    elif candidate.family == "extra":\n        estimator = ExtraTreesClassifier(\n            n_estimators=500,\n            max_depth=params["max_depth"],\n            min_samples_leaf=params["min_samples_leaf"],\n            max_features="sqrt",\n            class_weight=params["class_weight"],\n            n_jobs=-1,\n            random_state=SEED,\n        )\n        scaler = "passthrough"\n    elif candidate.family == "hgb":\n        estimator = HistGradientBoostingClassifier(\n            max_iter=350,\n            learning_rate=0.035,\n            max_leaf_nodes=params["max_leaf_nodes"],\n            min_samples_leaf=params["min_samples_leaf"],\n            l2_regularization=0.2,\n            random_state=SEED,\n        )\n        scaler = "passthrough"\n    elif candidate.family == "gb":\n        estimator = GradientBoostingClassifier(\n            n_estimators=300,\n            learning_rate=0.035,\n            max_depth=params["max_depth"],\n            min_samples_leaf=params["min_samples_leaf"],\n            subsample=0.8,\n            random_state=SEED,\n        )\n        scaler = "passthrough"\n    elif candidate.family == "svc":\n        estimator = SVC(\n            C=params["C"],\n            gamma=params["gamma"],\n            class_weight=params["class_weight"],\n            probability=True,\n            random_state=SEED,\n        )\n        scaler = RobustScaler()\n    else:\n        estimator = GaussianNB(var_smoothing=params["var_smoothing"])\n        scaler = RobustScaler()\n\n    steps = [("imputer", SimpleImputer(strategy="median", add_indicator=True))]\n    if candidate.winsorize:\n        steps.append(("winsor", QuantileClipper(0.01, 0.99)))\n    steps.extend([("scale", scaler), ("model", estimator)])\n    return Pipeline(steps)\n\n\ndef add_regime_features(\n    train: pd.DataFrame,\n    other: pd.DataFrame,\n    regime_columns: list[str],\n    components: int,\n):\n    if not components:\n        return train, other, []\n    unique_train = train[["quarter_index"] + regime_columns].drop_duplicates("quarter_index")\n    imputer = SimpleImputer(strategy="median")\n    scaler = StandardScaler()\n    x_unique = scaler.fit_transform(imputer.fit_transform(unique_train[regime_columns]))\n    model = GaussianMixture(\n        n_components=components,\n        covariance_type="diag",\n        reg_covar=1e-4,\n        n_init=10,\n        random_state=SEED,\n    )\n    model.fit(x_unique)\n    output_columns = [f"regime__prob_{i}" for i in range(components)]\n    result = []\n    for data in [train, other]:\n        transformed = scaler.transform(imputer.transform(data[regime_columns]))\n        probabilities = model.predict_proba(transformed)\n        enriched = data.copy()\n        for i, column in enumerate(output_columns):\n            enriched[column] = probabilities[:, i]\n        enriched["regime__uncertainty"] = 1.0 - probabilities.max(axis=1)\n        result.append(enriched)\n    output_columns.append("regime__uncertainty")\n    return result[0], result[1], output_columns\n\n\ndef fit_oof(\n    frame: pd.DataFrame,\n    feature_map: dict[str, list[str]],\n    regime_columns: list[str],\n    candidate: Candidate,\n) -> pd.DataFrame:\n    parts = []\n    for year in VALIDATION_YEARS:\n        train = frame[(frame.target_year < year) & frame[TARGET].notna()].copy()\n        validation = frame[(frame.target_year == year) & frame[TARGET].notna()].copy()\n        if len(train) < 150 or validation[TARGET].nunique() < 2:\n            continue\n        train, validation, regime_features = add_regime_features(\n            train, validation, regime_columns, candidate.gmm_components\n        )\n        features = feature_map[candidate.feature_mode] + regime_features\n        model = make_model(candidate)\n        sample_weight = None\n        if candidate.family == "gb" and candidate.params.get("risk_weight", 1.0) != 1.0:\n            sample_weight = np.where(\n                train[TARGET].astype(int).to_numpy() == 1,\n                candidate.params["risk_weight"],\n                1.0,\n            )\n        if sample_weight is None:\n            model.fit(train[features], train[TARGET].astype(int))\n        else:\n            model.fit(train[features], train[TARGET].astype(int), model__sample_weight=sample_weight)\n        probability = model.predict_proba(validation[features])[:, 1]\n        parts.append(\n            pd.DataFrame(\n                {\n                    "row_index": validation.index,\n                    "fold": year,\n                    "ticker": validation.ticker.to_numpy(),\n                    "quarter_index": validation.quarter_index.to_numpy(),\n                    "feature_quarter": validation.feature_quarter.to_numpy(),\n                    "target_quarter": validation.target_quarter.to_numpy(),\n                    "y": validation[TARGET].astype(int).to_numpy(),\n                    "probability": probability,\n                }\n            )\n        )\n    return pd.concat(parts, ignore_index=True)\n\n\ndef threshold_grid(oof: pd.DataFrame) -> pd.DataFrame:\n    probabilities = oof.probability.to_numpy()\n    pooled_pr_auc = average_precision_score(oof.y, probabilities)\n    pooled_roc_auc = roc_auc_score(oof.y, probabilities)\n    thresholds = np.unique(\n        np.r_[np.linspace(0.02, 0.98, 193), np.quantile(probabilities, np.linspace(0, 1, 101))]\n    )\n    fold_arrays = [\n        (group.y.to_numpy(), group.probability.to_numpy())\n        for _, group in oof.groupby("fold")\n    ]\n    rows = []\n    for threshold in thresholds:\n        pooled = threshold_metrics_fast(oof.y, probabilities, threshold)\n        pooled["pr_auc"] = pooled_pr_auc\n        pooled["roc_auc"] = pooled_roc_auc\n        folds = [threshold_metrics_fast(y, probability, threshold) for y, probability in fold_arrays]\n        fold_frame = pd.DataFrame(folds)\n        row = {\n            "threshold": float(threshold),\n            **pooled,\n            "mean_fold_accuracy": fold_frame.accuracy.mean(),\n            "mean_fold_risk_precision": fold_frame.risk_precision.mean(),\n            "mean_fold_risk_recall": fold_frame.risk_recall.mean(),\n            "mean_fold_norisk_precision": fold_frame.norisk_precision.mean(),\n            "mean_fold_norisk_recall": fold_frame.norisk_recall.mean(),\n            "worst_fold_risk_recall": fold_frame.risk_recall.min(),\n            "worst_fold_norisk_recall": fold_frame.norisk_recall.min(),\n            "std_fold_macro_f1": fold_frame.macro_f1.std(ddof=0),\n        }\n        constraints = {\n            "accuracy": 0.60,\n            "risk_recall": 0.50,\n            "norisk_precision": 0.70,\n            "norisk_recall": 0.50,\n        }\n        row["constraint_shortfall"] = sum(max(0.0, value - row[key]) for key, value in constraints.items())\n        row["validation_goal_pass"] = goal_pass(row)\n        row["selection_score"] = (\n            2.4 * row["risk_precision"]\n            + 0.8 * row["risk_recall"]\n            + 0.5 * row["norisk_recall"]\n            + 0.25 * row["accuracy"]\n            + 0.15 * row["macro_f1"]\n            + 0.15 * row["mean_fold_risk_precision"]\n            + 0.10 * row["mean_fold_risk_recall"]\n            - 4.0 * row["constraint_shortfall"]\n            - 0.15 * row["std_fold_macro_f1"]\n        )\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef select_operating_point(grid: pd.DataFrame) -> pd.Series:\n    eligible = grid[\n        (grid.accuracy >= 0.60)\n        & (grid.risk_recall >= 0.50)\n        & (grid.norisk_precision >= 0.70)\n        & (grid.norisk_recall >= 0.50)\n        & (grid.mean_fold_risk_recall >= 0.42)\n        & (grid.mean_fold_norisk_recall >= 0.50)\n    ]\n    pool = eligible if not eligible.empty else grid\n    return pool.sort_values(\n        ["validation_goal_pass", "constraint_shortfall", "selection_score", "risk_precision"],\n        ascending=[False, True, False, False],\n    ).iloc[0]\n\n\ndef build_ensembles(\n    oof_cache: dict[str, pd.DataFrame],\n    individual_results: pd.DataFrame,\n) -> tuple[pd.DataFrame, dict[str, dict]]:\n    ranked = individual_results.sort_values(\n        ["constraint_shortfall", "selection_score"], ascending=[True, False]\n    )\n    top_ids = []\n    family_mode_seen = {}\n    for row in ranked.itertuples():\n        key = (row.family, row.feature_mode)\n        if family_mode_seen.get(key, 0) >= 3:\n            continue\n        top_ids.append(row.candidate_id)\n        family_mode_seen[key] = family_mode_seen.get(key, 0) + 1\n        if len(top_ids) >= 16:\n            break\n\n    common = oof_cache[top_ids[0]][\n        ["row_index", "fold", "ticker", "quarter_index", "feature_quarter", "target_quarter", "y"]\n    ].copy()\n    probability_matrix = {}\n    for candidate_id in top_ids:\n        values = oof_cache[candidate_id].set_index("row_index").loc[common.row_index, "probability"]\n        probability_matrix[candidate_id] = values.to_numpy()\n    probabilities = pd.DataFrame(probability_matrix)\n    correlation = probabilities.corr()\n\n    definitions = {}\n    rows = []\n    ensemble_counter = 0\n\n    def evaluate(members, weights, method):\n        nonlocal ensemble_counter\n        ensemble_counter += 1\n        ensemble_id = f"ENS{ensemble_counter:03d}"\n        matrix = probabilities[list(members)].to_numpy()\n        if method == "rank_mean":\n            ranked_matrix = np.zeros_like(matrix)\n            for fold in sorted(common.fold.unique()):\n                mask = common.fold.to_numpy() == fold\n                for column in range(matrix.shape[1]):\n                    ranked_matrix[mask, column] = pd.Series(matrix[mask, column]).rank(pct=True).to_numpy()\n            ensemble_probability = np.average(ranked_matrix, axis=1, weights=weights)\n        else:\n            ensemble_probability = np.average(matrix, axis=1, weights=weights)\n        ensemble_oof = common.copy()\n        ensemble_oof["probability"] = ensemble_probability\n        grid = threshold_grid(ensemble_oof)\n        best = select_operating_point(grid)\n        definitions[ensemble_id] = {\n            "members": list(members),\n            "weights": list(weights),\n            "method": method,\n            "threshold": float(best.threshold),\n        }\n        rows.append({"ensemble_id": ensemble_id, "members": "|".join(members), "method": method, **best.to_dict()})\n\n    for left, right in combinations(top_ids[:12], 2):\n        if correlation.loc[left, right] > 0.985:\n            continue\n        for weight in [0.25, 0.5, 0.75]:\n            evaluate((left, right), (weight, 1.0 - weight), "probability_mean")\n        evaluate((left, right), (0.5, 0.5), "rank_mean")\n\n    for members in combinations(top_ids[:10], 3):\n        pair_correlations = [correlation.loc[a, b] for a, b in combinations(members, 2)]\n        if max(pair_correlations) > 0.98:\n            continue\n        evaluate(members, (1 / 3, 1 / 3, 1 / 3), "probability_mean")\n        evaluate(members, (1 / 3, 1 / 3, 1 / 3), "rank_mean")\n\n    return pd.DataFrame(rows), definitions\n\n\ndef final_probability(\n    frame: pd.DataFrame,\n    feature_map: dict[str, list[str]],\n    regime_columns: list[str],\n    candidate: Candidate,\n) -> tuple[np.ndarray, Pipeline, list[str], pd.DataFrame]:\n    train = frame[(frame.target_year < 2024) & frame[TARGET].notna()].copy()\n    test = frame[(frame.target_year >= 2024) & frame[TARGET].notna()].copy()\n    train, test, regime_features = add_regime_features(\n        train, test, regime_columns, candidate.gmm_components\n    )\n    features = feature_map[candidate.feature_mode] + regime_features\n    model = make_model(candidate)\n    sample_weight = None\n    if candidate.family == "gb" and candidate.params.get("risk_weight", 1.0) != 1.0:\n        sample_weight = np.where(\n            train[TARGET].astype(int).to_numpy() == 1,\n            candidate.params["risk_weight"],\n            1.0,\n        )\n    if sample_weight is None:\n        model.fit(train[features], train[TARGET].astype(int))\n    else:\n        model.fit(train[features], train[TARGET].astype(int), model__sample_weight=sample_weight)\n    return model.predict_proba(test[features])[:, 1], model, features, test\n\n\ndef main() -> None:\n    started = time.time()\n    for directory in [OUT, OUT / "validation", OUT / "test", OUT / "models", OUT / "notes"]:\n        directory.mkdir(parents=True, exist_ok=True)\n\n    base_manifest = json.loads(BASE_MANIFEST.read_text(encoding="utf-8"))\n    base_features = list(base_manifest["features"])\n    raw = pd.read_csv(INPUT)\n    frame, feature_map, regime_columns = build_source_guided_features(raw, base_features)\n    specs = candidate_specs()\n\n    oof_cache = {}\n    candidate_rows = []\n    for number, candidate in enumerate(specs, 1):\n        oof = fit_oof(frame, feature_map, regime_columns, candidate)\n        grid = threshold_grid(oof)\n        best = select_operating_point(grid)\n        oof_cache[candidate.candidate_id] = oof\n        candidate_rows.append(\n            {\n                **asdict(candidate),\n                "params": json.dumps(candidate.params),\n                "feature_count": len(feature_map[candidate.feature_mode]) + candidate.gmm_components + bool(candidate.gmm_components),\n                **best.to_dict(),\n            }\n        )\n        if number % 20 == 0:\n            print(f"individual candidate {number}/{len(specs)}", flush=True)\n\n    individual_results = pd.DataFrame(candidate_rows).drop(columns=["params"], errors="ignore")\n    individual_results = individual_results.sort_values(\n        ["constraint_shortfall", "selection_score"], ascending=[True, False]\n    )\n    individual_results.to_csv(OUT / "validation" / "individual_results.csv", index=False)\n\n    ensemble_results, ensemble_definitions = build_ensembles(oof_cache, individual_results)\n    ensemble_results = ensemble_results.sort_values(\n        ["constraint_shortfall", "selection_score"], ascending=[True, False]\n    )\n    ensemble_results.to_csv(OUT / "validation" / "ensemble_results.csv", index=False)\n\n    validation_choices = []\n    for row in individual_results.head(8).itertuples():\n        validation_choices.append(\n            {\n                "choice_id": row.candidate_id,\n                "choice_type": "individual",\n                "threshold": row.threshold,\n                "validation_score": row.selection_score,\n                "validation_constraint_shortfall": row.constraint_shortfall,\n                "validation_risk_precision": row.risk_precision,\n                "validation_risk_recall": row.risk_recall,\n                "validation_norisk_precision": row.norisk_precision,\n                "validation_norisk_recall": row.norisk_recall,\n            }\n        )\n    for row in ensemble_results.head(12).itertuples():\n        validation_choices.append(\n            {\n                "choice_id": row.ensemble_id,\n                "choice_type": "ensemble",\n                "threshold": row.threshold,\n                "validation_score": row.selection_score,\n                "validation_constraint_shortfall": row.constraint_shortfall,\n                "validation_risk_precision": row.risk_precision,\n                "validation_risk_recall": row.risk_recall,\n                "validation_norisk_precision": row.norisk_precision,\n                "validation_norisk_recall": row.norisk_recall,\n            }\n        )\n    shortlist = pd.DataFrame(validation_choices).sort_values(\n        ["validation_constraint_shortfall", "validation_score"], ascending=[True, False]\n    )\n    shortlist = shortlist.drop_duplicates("choice_id").head(12).reset_index(drop=True)\n    shortlist.to_csv(OUT / "validation" / "shortlist.csv", index=False)\n\n    spec_map = {candidate.candidate_id: candidate for candidate in specs}\n    required_ids = set()\n    for choice in shortlist.itertuples():\n        if choice.choice_type == "individual":\n            required_ids.add(choice.choice_id)\n        else:\n            required_ids.update(ensemble_definitions[choice.choice_id]["members"])\n\n    test_probabilities = {}\n    fitted_models = {}\n    feature_lists = {}\n    test_reference = None\n    for candidate_id in sorted(required_ids):\n        probability, model, features, test = final_probability(\n            frame, feature_map, regime_columns, spec_map[candidate_id]\n        )\n        test_probabilities[candidate_id] = probability\n        fitted_models[candidate_id] = model\n        feature_lists[candidate_id] = features\n        test_reference = test\n\n    test_rows = []\n    prediction_parts = []\n    for choice in shortlist.itertuples():\n        if choice.choice_type == "individual":\n            probability = test_probabilities[choice.choice_id]\n        else:\n            definition = ensemble_definitions[choice.choice_id]\n            matrix = np.column_stack([test_probabilities[x] for x in definition["members"]])\n            if definition["method"] == "rank_mean":\n                ranked = np.zeros_like(matrix)\n                for quarter in sorted(test_reference.quarter_index.unique()):\n                    mask = test_reference.quarter_index.to_numpy() == quarter\n                    for column in range(matrix.shape[1]):\n                        ranked[mask, column] = pd.Series(matrix[mask, column]).rank(pct=True).to_numpy()\n                probability = np.average(ranked, axis=1, weights=definition["weights"])\n            else:\n                probability = np.average(matrix, axis=1, weights=definition["weights"])\n        prediction = (probability >= choice.threshold).astype(int)\n        metrics = metric_row(test_reference[TARGET], prediction, probability)\n        test_rows.append(\n            {\n                "choice_id": choice.choice_id,\n                "choice_type": choice.choice_type,\n                "threshold": choice.threshold,\n                "validation_score": choice.validation_score,\n                "validation_constraint_shortfall": choice.validation_constraint_shortfall,\n                **metrics,\n                "goal_pass": goal_pass(metrics),\n            }\n        )\n        prediction_parts.append(\n            pd.DataFrame(\n                {\n                    "choice_id": choice.choice_id,\n                    "ticker": test_reference.ticker.to_numpy(),\n                    "feature_quarter": test_reference.feature_quarter.to_numpy(),\n                    "target_quarter": test_reference.target_quarter.to_numpy(),\n                    "target_year": test_reference.target_year.to_numpy(),\n                    TARGET: test_reference[TARGET].astype(int).to_numpy(),\n                    "risk_probability": probability,\n                    "prediction": prediction,\n                }\n            )\n        )\n\n    test_results = pd.DataFrame(test_rows).sort_values(\n        ["goal_pass", "risk_precision", "risk_recall", "accuracy"],\n        ascending=[False, False, False, False],\n    )\n    test_results.to_csv(OUT / "test" / "shortlist_metrics.csv", index=False)\n    pd.concat(prediction_parts, ignore_index=True).to_csv(\n        OUT / "test" / "shortlist_predictions.csv", index=False\n    )\n\n    selected = shortlist.iloc[0]\n    if selected.choice_type == "individual":\n        selected_members = [selected.choice_id]\n        selected_definition = {\n            "members": selected_members,\n            "weights": [1.0],\n            "method": "individual",\n            "threshold": float(selected.threshold),\n        }\n    else:\n        selected_definition = ensemble_definitions[selected.choice_id]\n        selected_members = selected_definition["members"]\n    for candidate_id in selected_members:\n        joblib.dump(fitted_models[candidate_id], OUT / "models" / f"{candidate_id}.joblib")\n\n    manifest = {\n        "input": str(INPUT),\n        "real_data_only": True,\n        "target": TARGET,\n        "validation_years": list(VALIDATION_YEARS),\n        "test_period": "target_year >= 2024",\n        "individual_candidate_count": len(specs),\n        "ensemble_candidate_count": len(ensemble_results),\n        "feature_modes": {key: len(value) for key, value in feature_map.items()},\n        "regime_columns": regime_columns,\n        "validation_selected_choice": selected.to_dict(),\n        "selected_definition": selected_definition,\n        "selected_member_specs": {member: asdict(spec_map[member]) for member in selected_members},\n        "selected_member_features": {member: feature_lists[member] for member in selected_members},\n        "runtime_seconds": time.time() - started,\n    }\n    (OUT / "models" / "manifest.json").write_text(\n        json.dumps(manifest, indent=2, default=str), encoding="utf-8"\n    )\n\n    best_test = test_results.iloc[0]\n    selected_test = test_results[test_results.choice_id == selected.choice_id].iloc[0]\n    lines = [\n        "# Source-guided financial deterioration experiments",\n        "",\n        "## Protocol",\n        "",\n        "- Real quarterly financial data only; no simulated or synthetic rows.",\n        "- Trajectory, contextual-regime, robust-outlier, and diversity-aware ensemble directions were tested.",\n        "- Model and threshold selection used expanding target-year folds 2020-2023.",\n        "- The 2024+ holdout was evaluated only after the validation shortlist was frozen.",\n        "",\n        "## Validation-selected result",\n        "",\n        f"- Choice: `{selected.choice_id}` ({selected.choice_type}).",\n        f"- Holdout accuracy: {selected_test.accuracy:.2%}.",\n        f"- Risk precision: {selected_test.risk_precision:.2%}.",\n        f"- Risk recall: {selected_test.risk_recall:.2%}.",\n        f"- NoRisk precision: {selected_test.norisk_precision:.2%}.",\n        f"- NoRisk recall: {selected_test.norisk_recall:.2%}.",\n        f"- Goal passed: **{bool(selected_test.goal_pass)}**.",\n        "",\n        "## Best fixed-shortlist holdout result",\n        "",\n        f"- Choice: `{best_test.choice_id}` ({best_test.choice_type}).",\n        f"- Holdout accuracy: {best_test.accuracy:.2%}.",\n        f"- Risk precision: {best_test.risk_precision:.2%}.",\n        f"- Risk recall: {best_test.risk_recall:.2%}.",\n        f"- NoRisk precision: {best_test.norisk_precision:.2%}.",\n        f"- NoRisk recall: {best_test.norisk_recall:.2%}.",\n        f"- Goal passed: **{bool(best_test.goal_pass)}**.",\n        "",\n        "The best-shortlist row is diagnostic only; the validation-selected row remains the honest primary result.",\n    ]\n    (OUT / "notes" / "FINAL_REPORT.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    print(test_results.to_json(orient="records", indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'run_reportnorm_expansion.py': 'from __future__ import annotations\n\nimport json\nimport time\nfrom dataclasses import asdict\nfrom pathlib import Path\n\nimport joblib\nimport numpy as np\nimport pandas as pd\n\nimport run_source_guided_experiments as sg\n\nTARGET = sg.TARGET\nOUT = Path(__file__).resolve().parent / "reportnorm_expansion_output"\n\n\ndef safe_divide(a: pd.Series, b: pd.Series) -> pd.Series:\n    return (a / b.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)\n\n\ndef add_reportnorm_features(frame: pd.DataFrame, feature_map: dict[str, list[str]]):\n    q = frame.sort_values(["ticker", "quarter_index"]).copy()\n    original_index = q.index\n    reportnorm = [c for c in q.columns if c.startswith("reportnorm_") and c.endswith("_million_vnd")]\n    liquid = [f"reportnorm_{code}_million_vnd" for code in (4310, 4311, 4312, 4313)]\n    short_liability = [f"reportnorm_{code}_million_vnd" for code in (4318, 4319, 4321, 4322, 4323, 4324)]\n\n    raw_features = []\n    ratio_features = []\n    trend_features = []\n    rank_features = []\n    missing_features = []\n    for column in reportnorm:\n        q[column] = pd.to_numeric(q[column], errors="coerce")\n        short = column.replace("_million_vnd", "")\n        lag1 = sg.contiguous_lag(q, column, 1)\n        lag4 = sg.contiguous_lag(q, column, 4)\n        q[f"rn__{short}__qoq_pct"] = safe_divide(q[column] - lag1, lag1.abs())\n        q[f"rn__{short}__yoy_pct"] = safe_divide(q[column] - lag4, lag4.abs())\n        q[f"rn__{short}__assets_ratio"] = safe_divide(q[column], q.total_assets_million_vnd)\n        q[f"rn__{short}__quarter_rank"] = q.groupby("quarter_index")[column].rank(pct=True)\n        q[f"rn__{short}__missing"] = q[column].isna().astype(float)\n        ratio = q[f"rn__{short}__assets_ratio"]\n        q[f"rn__{short}__ratio_mean4"] = sg.contiguous_rolling(q, ratio, 4, 2, "mean")\n        q[f"rn__{short}__ratio_std4"] = sg.contiguous_rolling(q, ratio, 4, 2, "std")\n        raw_features.append(column)\n        trend_features.extend([f"rn__{short}__qoq_pct", f"rn__{short}__yoy_pct"])\n        ratio_features.extend(\n            [\n                f"rn__{short}__assets_ratio",\n                f"rn__{short}__ratio_mean4",\n                f"rn__{short}__ratio_std4",\n            ]\n        )\n        rank_features.append(f"rn__{short}__quarter_rank")\n        missing_features.append(f"rn__{short}__missing")\n\n    q["rn__liquid_sum"] = q[liquid].sum(axis=1, min_count=2)\n    q["rn__short_liability_sum"] = q[short_liability].sum(axis=1, min_count=3)\n    q["rn__current_ratio"] = safe_divide(q.rn__liquid_sum, q.rn__short_liability_sum)\n    q["rn__liquid_to_assets"] = safe_divide(q.rn__liquid_sum, q.total_assets_million_vnd)\n    q["rn__short_liability_to_assets"] = safe_divide(\n        q.rn__short_liability_sum, q.total_assets_million_vnd\n    )\n    q["rn__coverage"] = q[reportnorm].notna().mean(axis=1)\n\n    aggregate_features = [\n        "rn__liquid_sum",\n        "rn__short_liability_sum",\n        "rn__current_ratio",\n        "rn__liquid_to_assets",\n        "rn__short_liability_to_assets",\n        "rn__coverage",\n    ]\n    for column in aggregate_features[:-1]:\n        lag1 = sg.contiguous_lag(q, column, 1)\n        lag4 = sg.contiguous_lag(q, column, 4)\n        q[f"{column}__qoq_pct"] = safe_divide(q[column] - lag1, lag1.abs())\n        q[f"{column}__yoy_pct"] = safe_divide(q[column] - lag4, lag4.abs())\n        q[f"{column}__quarter_rank"] = q.groupby("quarter_index")[column].rank(pct=True)\n        q[f"{column}__mean4"] = sg.contiguous_rolling(q, q[column], 4, 2, "mean")\n        q[f"{column}__std4"] = sg.contiguous_rolling(q, q[column], 4, 2, "std")\n        aggregate_features.extend(\n            [\n                f"{column}__qoq_pct",\n                f"{column}__yoy_pct",\n                f"{column}__quarter_rank",\n                f"{column}__mean4",\n                f"{column}__std4",\n            ]\n        )\n\n    q["rnctx__current_ratio_median"] = q.groupby("quarter_index").rn__current_ratio.transform("median")\n    q["rnctx__current_ratio_std"] = q.groupby("quarter_index").rn__current_ratio.transform("std")\n    q["rnctx__liquid_to_assets_median"] = q.groupby("quarter_index").rn__liquid_to_assets.transform("median")\n    q["rnctx__short_liability_to_assets_median"] = q.groupby(\n        "quarter_index"\n    ).rn__short_liability_to_assets.transform("median")\n    context_features = [\n        "rnctx__current_ratio_median",\n        "rnctx__current_ratio_std",\n        "rnctx__liquid_to_assets_median",\n        "rnctx__short_liability_to_assets_median",\n    ]\n\n    component_features = list(\n        dict.fromkeys(\n            raw_features\n            + ratio_features\n            + trend_features\n            + rank_features\n            + missing_features\n            + aggregate_features\n            + context_features\n        )\n    )\n    stable_features = list(\n        dict.fromkeys(\n            ratio_features\n            + trend_features\n            + rank_features\n            + missing_features\n            + [c for c in aggregate_features if "_sum" not in c or c.endswith(("pct", "rank"))]\n            + context_features\n        )\n    )\n    ticker_features = [c for c in feature_map["source_full_ticker"] if c.startswith("ticker__")]\n    expanded_map = {\n        "reportnorm_stable": list(dict.fromkeys(feature_map["source_compact"] + stable_features)),\n        "reportnorm_full": list(dict.fromkeys(feature_map["source_full"] + component_features)),\n        "reportnorm_ticker": list(\n            dict.fromkeys(feature_map["source_compact"] + stable_features + ticker_features)\n        ),\n    }\n    return q.loc[original_index], expanded_map, component_features\n\n\ndef candidates():\n    rows = []\n    counter = 0\n\n    def add(family, mode, params, winsor=False):\n        nonlocal counter\n        counter += 1\n        rows.append(sg.Candidate(f"RN{counter:03d}", family, mode, params, winsor, 0))\n\n    for mode in ["reportnorm_stable", "reportnorm_full", "reportnorm_ticker"]:\n        for c in [0.003, 0.01, 0.03, 0.1]:\n            for weight in [None, "balanced", {0: 1.0, 1: 1.5}]:\n                add("logreg", mode, {"C": c, "class_weight": weight}, winsor=True)\n        for depth, leaf, weight in [\n            (4, 8, None),\n            (6, 12, None),\n            (8, 20, None),\n            (4, 10, "balanced"),\n        ]:\n            add("rf", mode, {"max_depth": depth, "min_samples_leaf": leaf, "class_weight": weight})\n            add("extra", mode, {"max_depth": depth, "min_samples_leaf": leaf, "class_weight": weight})\n        for leaves, leaf in [(5, 12), (9, 20), (15, 30)]:\n            add("hgb", mode, {"max_leaf_nodes": leaves, "min_samples_leaf": leaf})\n        for depth, leaf, risk_weight in [(1, 12, 1.0), (1, 20, 1.5), (2, 20, 1.0)]:\n            add(\n                "gb",\n                mode,\n                {"max_depth": depth, "min_samples_leaf": leaf, "risk_weight": risk_weight},\n            )\n        for c, gamma, weight in [\n            (0.2, "scale", None),\n            (1.0, "scale", None),\n            (1.0, 0.01, "balanced"),\n        ]:\n            add("svc", mode, {"C": c, "gamma": gamma, "class_weight": weight}, winsor=True)\n    return rows\n\n\ndef main():\n    started = time.time()\n    for directory in [OUT, OUT / "validation", OUT / "test", OUT / "models", OUT / "notes"]:\n        directory.mkdir(parents=True, exist_ok=True)\n\n    base_manifest = json.loads(sg.BASE_MANIFEST.read_text(encoding="utf-8"))\n    raw = pd.read_csv(sg.INPUT)\n    frame, original_map, regime_columns = sg.build_source_guided_features(\n        raw, base_manifest["features"]\n    )\n    frame, feature_map, reportnorm_features = add_reportnorm_features(frame, original_map)\n    specs = candidates()\n    oof_cache = {}\n    rows = []\n    for number, candidate in enumerate(specs, 1):\n        oof = sg.fit_oof(frame, feature_map, [], candidate)\n        grid = sg.threshold_grid(oof)\n        best = sg.select_operating_point(grid)\n        oof_cache[candidate.candidate_id] = oof\n        rows.append(\n            {\n                **asdict(candidate),\n                "params": json.dumps(candidate.params),\n                "feature_count": len(feature_map[candidate.feature_mode]),\n                **best.to_dict(),\n            }\n        )\n        if number % 20 == 0:\n            print(f"reportnorm candidate {number}/{len(specs)}", flush=True)\n\n    individual = pd.DataFrame(rows).sort_values(\n        ["constraint_shortfall", "selection_score"], ascending=[True, False]\n    )\n    individual.to_csv(OUT / "validation" / "individual_results.csv", index=False)\n    ensembles, ensemble_definitions = sg.build_ensembles(oof_cache, individual)\n    ensembles = ensembles.sort_values(\n        ["constraint_shortfall", "selection_score"], ascending=[True, False]\n    )\n    ensembles.to_csv(OUT / "validation" / "ensemble_results.csv", index=False)\n\n    choices = []\n    seen = set()\n    combined = []\n    for row in individual.head(12).itertuples():\n        combined.append(\n            {\n                "choice_id": row.candidate_id,\n                "choice_type": "individual",\n                "threshold": row.threshold,\n                "validation_score": row.selection_score,\n                "validation_constraint_shortfall": row.constraint_shortfall,\n                "validation_risk_precision": row.risk_precision,\n                "validation_risk_recall": row.risk_recall,\n            }\n        )\n    for row in ensembles.head(20).itertuples():\n        combined.append(\n            {\n                "choice_id": row.ensemble_id,\n                "choice_type": "ensemble",\n                "threshold": row.threshold,\n                "validation_score": row.selection_score,\n                "validation_constraint_shortfall": row.constraint_shortfall,\n                "validation_risk_precision": row.risk_precision,\n                "validation_risk_recall": row.risk_recall,\n            }\n        )\n    for row in sorted(\n        combined,\n        key=lambda x: (x["validation_constraint_shortfall"], -x["validation_score"]),\n    ):\n        if row["choice_id"] in seen:\n            continue\n        choices.append(row)\n        seen.add(row["choice_id"])\n        if len(choices) >= 14:\n            break\n    shortlist = pd.DataFrame(choices)\n    shortlist.to_csv(OUT / "validation" / "shortlist.csv", index=False)\n\n    spec_map = {candidate.candidate_id: candidate for candidate in specs}\n    required = set()\n    for choice in shortlist.itertuples():\n        if choice.choice_type == "individual":\n            required.add(choice.choice_id)\n        else:\n            required.update(ensemble_definitions[choice.choice_id]["members"])\n    probabilities = {}\n    models = {}\n    features_used = {}\n    test_reference = None\n    for candidate_id in sorted(required):\n        probability, model, features, test = sg.final_probability(\n            frame, feature_map, [], spec_map[candidate_id]\n        )\n        probabilities[candidate_id] = probability\n        models[candidate_id] = model\n        features_used[candidate_id] = features\n        test_reference = test\n\n    test_rows = []\n    prediction_parts = []\n    for choice in shortlist.itertuples():\n        if choice.choice_type == "individual":\n            probability = probabilities[choice.choice_id]\n        else:\n            definition = ensemble_definitions[choice.choice_id]\n            matrix = np.column_stack([probabilities[x] for x in definition["members"]])\n            if definition["method"] == "rank_mean":\n                ranked = np.zeros_like(matrix)\n                for quarter in test_reference.target_quarter.unique():\n                    mask = test_reference.target_quarter.to_numpy() == quarter\n                    for column in range(matrix.shape[1]):\n                        ranked[mask, column] = pd.Series(matrix[mask, column]).rank(pct=True).to_numpy()\n                probability = np.average(ranked, axis=1, weights=definition["weights"])\n            else:\n                probability = np.average(matrix, axis=1, weights=definition["weights"])\n        prediction = (probability >= choice.threshold).astype(int)\n        metrics = sg.metric_row(test_reference[TARGET], prediction, probability)\n        test_rows.append(\n            {\n                "choice_id": choice.choice_id,\n                "choice_type": choice.choice_type,\n                "threshold": choice.threshold,\n                "validation_risk_precision": choice.validation_risk_precision,\n                "validation_risk_recall": choice.validation_risk_recall,\n                **metrics,\n                "goal_pass": sg.goal_pass(metrics),\n            }\n        )\n        prediction_parts.append(\n            pd.DataFrame(\n                {\n                    "choice_id": choice.choice_id,\n                    "ticker": test_reference.ticker.to_numpy(),\n                    "target_quarter": test_reference.target_quarter.to_numpy(),\n                    "target_year": test_reference.target_year.to_numpy(),\n                    TARGET: test_reference[TARGET].astype(int).to_numpy(),\n                    "risk_probability": probability,\n                    "prediction": prediction,\n                }\n            )\n        )\n\n    test_results = pd.DataFrame(test_rows).sort_values(\n        ["goal_pass", "risk_precision", "risk_recall", "accuracy"],\n        ascending=[False, False, False, False],\n    )\n    test_results.to_csv(OUT / "test" / "shortlist_metrics.csv", index=False)\n    pd.concat(prediction_parts, ignore_index=True).to_csv(\n        OUT / "test" / "shortlist_predictions.csv", index=False\n    )\n\n    selected = shortlist.iloc[0]\n    if selected.choice_type == "individual":\n        selected_members = [selected.choice_id]\n    else:\n        selected_members = ensemble_definitions[selected.choice_id]["members"]\n    for member in selected_members:\n        joblib.dump(models[member], OUT / "models" / f"{member}.joblib")\n\n    manifest = {\n        "real_data_only": True,\n        "source": str(sg.INPUT),\n        "reportnorm_feature_count": len(reportnorm_features),\n        "feature_modes": {key: len(value) for key, value in feature_map.items()},\n        "candidate_count": len(specs),\n        "ensemble_count": len(ensembles),\n        "validation_selected": selected.to_dict(),\n        "runtime_seconds": time.time() - started,\n    }\n    (OUT / "models" / "manifest.json").write_text(\n        json.dumps(manifest, indent=2, default=str), encoding="utf-8"\n    )\n\n    selected_test = test_results[test_results.choice_id == selected.choice_id].iloc[0]\n    best_test = test_results.iloc[0]\n    lines = [\n        "# ReportNorm expansion experiments",\n        "",\n        "- Ten real Vietstock ReportNorm liquidity and short-liability components were added.",\n        "- Features include levels, asset ratios, QoQ/YoY change, rolling stability, missing flags, and cross-sectional ranks.",\n        "- Selection used expanding 2020-2023 validation; 2024+ was evaluated after shortlist freeze.",\n        "",\n        "## Validation-selected",\n        f"- Choice: `{selected.choice_id}`.",\n        f"- Accuracy: {selected_test.accuracy:.2%}.",\n        f"- Risk precision: {selected_test.risk_precision:.2%}.",\n        f"- Risk recall: {selected_test.risk_recall:.2%}.",\n        f"- NoRisk precision: {selected_test.norisk_precision:.2%}.",\n        f"- NoRisk recall: {selected_test.norisk_recall:.2%}.",\n        f"- Goal passed: **{bool(selected_test.goal_pass)}**.",\n        "",\n        "## Best frozen-shortlist diagnostic",\n        f"- Choice: `{best_test.choice_id}`.",\n        f"- Accuracy: {best_test.accuracy:.2%}.",\n        f"- Risk precision: {best_test.risk_precision:.2%}.",\n        f"- Risk recall: {best_test.risk_recall:.2%}.",\n        f"- NoRisk precision: {best_test.norisk_precision:.2%}.",\n        f"- NoRisk recall: {best_test.norisk_recall:.2%}.",\n        f"- Goal passed: **{bool(best_test.goal_pass)}**.",\n    ]\n    (OUT / "notes" / "FINAL_REPORT.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n    print(test_results.to_json(orient="records", indent=2), flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'}
(RUNTIME_DIR / "canonical_sources").mkdir(parents=True, exist_ok=True)
for filename, source in EMBEDDED_SOURCES.items():
    target = RUNTIME_DIR / filename if filename.startswith("build_") else RUNTIME_DIR / "canonical_sources" / filename
    target.write_text(source, encoding="utf-8")
print("Embedded release files prepared:", len(EMBEDDED_SOURCES))


In [ ]:
env = os.environ.copy()
env["PYTHONUTF8"] = "1"
env["PYTHONIOENCODING"] = "utf-8"
command = [
    sys.executable,
    str(RUNTIME_DIR / "build_earnings_risk_release.py"),
    "--raw", str(RAW_PATH),
    "--canonical-source-dir", str(RUNTIME_DIR / "canonical_sources"),
    "--project-data-root", str(PROJECT_LAYERS),
    "--bundle-export-dir", str(EXPORT_DIR),
]
print("Running frozen release builder...")
subprocess.run(command, cwd=RUNTIME_DIR, env=env, check=True)


In [ ]:
import hashlib
import json
import pandas as pd

common_path = EXPORT_DIR / "common_master.csv"
common = pd.read_csv(common_path)
frozen = pd.read_csv(FROZEN_MASTER)
summary = {
    "rows": len(common),
    "columns": common.shape[1],
    "feature_258": len(pd.read_csv(EXPORT_DIR / "feature_list_258.csv")),
    "feature_309": len(pd.read_csv(EXPORT_DIR / "feature_list_309.csv")),
    "schema_matches_frozen": list(common.columns) == list(frozen.columns),
    "row_keys_match_frozen": set(common.row_key) == set(frozen.row_key),
    "risk10_labels_match_frozen": dict(zip(common.row_key, common.risk10_label)) == dict(zip(frozen.row_key, frozen.risk10_label)),
}
display(pd.DataFrame([summary]))
(OUTPUT_DIR / "NOTEBOOK_ACCEPTANCE.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print("Generated master:", common_path)
